# Myanmar Voice Clone — Gradio Colab

Notebook này là bản **voice clone only** có giao diện Gradio.

Không dùng prompt voice preset `Myanmar 1–4`. Bạn chỉ chọn/upload **reference voice** thật, nhập text Myanmar, chỉnh thông số, rồi generate.

Luồng chạy:

1. Mount Google Drive.
2. Cài thư viện.
3. Tạo lại package pipeline từ các file `.py`.
4. Mở Gradio UI.
5. Chọn voice trong thư mục Drive hoặc upload voice mới.
6. Generate và nghe output trực tiếp.


In [15]:
#@title 1) Kiểm tra GPU và mount Google Drive
import os, sys, json, shutil, subprocess, textwrap, importlib, gc
from pathlib import Path

try:
    import torch
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch chưa sẵn sàng:", e)

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DRIVE_DIR = Path("/content/drive/MyDrive/OmniVoice_Myanmar_Gradio")
REF_DIR = PROJECT_DRIVE_DIR / "reference_voices"
OUTPUT_DIR = PROJECT_DRIVE_DIR / "outputs"
CACHE_DIR = PROJECT_DRIVE_DIR / "voice_clone_cache"
WORK_DIR = PROJECT_DRIVE_DIR / "work"

for d in [PROJECT_DRIVE_DIR, REF_DIR, OUTPUT_DIR, CACHE_DIR, WORK_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_DRIVE_DIR:", PROJECT_DRIVE_DIR)
print("REF_DIR:", REF_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CACHE_DIR:", CACHE_DIR)


Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_DRIVE_DIR: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio
REF_DIR: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio/reference_voices
OUTPUT_DIR: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio/outputs
CACHE_DIR: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio/voice_clone_cache


In [16]:
#@title 2) Cài thư viện cần thiết
# Nếu pip install omnivoice lỗi, bật INSTALL_FROM_GITHUB = True rồi chạy lại cell.
INSTALL_FROM_GITHUB = False  #@param {type:"boolean"}

base_packages = [
    "soundfile",
    "librosa",
    "pedalboard",
    "gradio",
    "accelerate",
    "safetensors",
    "transformers",
    "huggingface_hub",
]

cmd = [sys.executable, "-m", "pip", "install", "-q", "-U"] + base_packages
print("Installing base packages...")
subprocess.check_call(cmd)

if INSTALL_FROM_GITHUB:
    print("Installing OmniVoice from GitHub...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "git+https://github.com/k2-fsa/OmniVoice.git"])
else:
    print("Installing OmniVoice from PyPI...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "omnivoice"])

print("Done.")


Installing base packages...
Installing OmniVoice from PyPI...
Done.


In [17]:
#@title 3) Tạo package pipeline từ các file bạn gửi
from pathlib import Path

PKG_ROOT = Path("/content/myanmar_voice_pipeline_src")
PKG_DIR = PKG_ROOT / "myanmar_voice_pipeline"
PKG_DIR.mkdir(parents=True, exist_ok=True)
(PKG_DIR / "__init__.py").write_text("", encoding="utf-8")

module_sources = {
    'presets.py': 'from __future__ import annotations\n\nimport os\nimport unicodedata\nfrom pathlib import Path\nfrom typing import Any, Dict, List, Optional\n\nBASE_OMNI_CONFIG: Dict[str, Any] = {\n    "num_step": 32,\n    "guidance_scale": 3.8,\n    "t_shift": 0.1,\n    "layer_penalty_factor": 10.0,\n    "position_temperature": 0.0,\n    "class_temperature": 0.0,\n    "denoise": True,\n    "preprocess_prompt": True,\n    "postprocess_output": False,\n    "audio_chunk_duration": 0.0,\n    "audio_chunk_threshold": 60.0,\n    "speed": 1.0,\n    "pitch_shift": 1.0,\n    "join_silence_ms": 120,\n    "trailing_silence_ms": 250,\n    "min_join_silence_ms": 45,\n    "max_segment_chars": None,\n    "instruct": None,\n}\n\nLANGUAGE_LABELS: Dict[str, str] = {\n    "vi": "Vietnamese",\n    "lo": "Lao",\n    "km": "Khmer",\n    "th": "Thai",\n    "my": "Myanmar",\n}\n\nLANGUAGE_PRESETS: Dict[str, Dict[str, Any]] = {\n    "vi": {"label": "Vietnamese", "instruct": "male, young adult"},\n    "lo": {"label": "Lao", "instruct": "female, moderate pitch"},\n    "km": {"label": "Khmer", "instruct": "male, moderate pitch"},\n    "th": {"label": "Thai", "instruct": "female, high pitch"},\n    "my": {"label": "Myanmar", "instruct": "female, young adult"},\n}\n\nDEPLOY_LANGUAGE_CONFIGS: Dict[str, Dict[str, Any]] = {\n    "vi": {\n        "num_step": 32,\n        "guidance_scale": 3.9,\n        "speed": 1.0,\n        "pitch_shift": 1.0,\n        "join_silence_ms": 110,\n        "trailing_silence_ms": 220,\n        "preprocess_prompt": True,\n        "max_segment_chars": 110,\n    },\n    "lo": {\n        "num_step": 32,\n        "guidance_scale": 3.8,\n        "speed": 1.0,\n        "pitch_shift": 1.0,\n        "join_silence_ms": 130,\n        "trailing_silence_ms": 260,\n        "preprocess_prompt": True,\n    },\n    "km": {\n        "num_step": 32,\n        "guidance_scale": 3.8,\n        "speed": 1.0,\n        "pitch_shift": 1.0,\n        "join_silence_ms": 130,\n        "trailing_silence_ms": 260,\n        "preprocess_prompt": True,\n    },\n    "th": {\n        "num_step": 32,\n        "guidance_scale": 3.9,\n        "speed": 1.0,\n        "pitch_shift": 1.0,\n        "join_silence_ms": 120,\n        "trailing_silence_ms": 240,\n        "preprocess_prompt": True,\n    },\n    "my": {\n        "num_step": 36,\n        "guidance_scale": 4.0,\n        "speed": 1.0,\n        "pitch_shift": 1.0,\n        "join_silence_ms": 95,\n        "trailing_silence_ms": 180,\n        "preprocess_prompt": True,\n        "max_segment_chars": 90,\n    },\n}\nEMOTION_PRESETS_BY_LANG: Dict[str, Dict[str, Dict[str, Any]]] = {\n    "vi": {\n        "Mặc định": {\n            "overrides": {},\n            "gain_db": 0.0,\n        },\n\n        # Vui: nhanh hơn nhẹ, sáng hơn nhẹ, không đẩy gain quá cao\n        "Vui vẻ (Happy)": {\n            "overrides": {\n                "pitch_shift": 1.01,\n                "speed": 1.045,\n                "join_silence_ms": 100,\n                "trailing_silence_ms": 190,\n            },\n            "gain_db": 0.55,\n        },\n\n        # Buồn: chậm, ngắt dài hơn, giảm gain để giọng mềm và trầm hơn\n        "Buồn bã (Sad)": {\n            "overrides": {\n                "pitch_shift": 0.985,\n                "speed": 0.92,\n                "join_silence_ms": 165,\n                "trailing_silence_ms": 430,\n            },\n            "gain_db": -1.1,\n        },\n\n        # Hào hứng: nhanh rõ nhưng không pitch quá cao để tránh rè\n        "Hào hứng (Excited)": {\n            "overrides": {\n                "pitch_shift": 1.015,\n                "speed": 1.075,\n                "join_silence_ms": 80,\n                "trailing_silence_ms": 155,\n            },\n            "gain_db": 0.8,\n        },\n\n        # Giận: tốc độ nhanh, khoảng nghỉ ngắn, pitch hơi thấp để chắc giọng\n        "Giận dữ (Angry)": {\n            "overrides": {\n                "pitch_shift": 0.985,\n                "speed": 1.06,\n                "join_silence_ms": 75,\n                "trailing_silence_ms": 145,\n            },\n            "gain_db": 0.75,\n        },\n\n        # Nhẹ nhàng: chậm, nghỉ mềm, gain âm nhẹ\n        "Nhẹ nhàng (Gentle)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.94,\n                "join_silence_ms": 140,\n                "trailing_silence_ms": 310,\n            },\n            "gain_db": -0.55,\n        },\n    },\n\n    "lo": {\n        "Mặc định": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.985,\n                "join_silence_ms": 136,\n                "trailing_silence_ms": 268,\n                "num_step": 32,\n                "guidance_scale": 4.0,\n            },\n            "gain_db": 0.0,\n        },\n\n        "Vui vẻ (Happy)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 1.005,\n                "join_silence_ms": 128,\n                "trailing_silence_ms": 248,\n                "num_step": 32,\n                "guidance_scale": 3.95,\n            },\n            "gain_db": 0.18,\n        },\n\n        "Ngạc nhiên (Surprised)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 1.012,\n                "join_silence_ms": 118,\n                "trailing_silence_ms": 230,\n                "num_step": 32,\n                "guidance_scale": 4.05,\n            },\n            "gain_db": 0.22,\n        },\n\n        "Buồn bã (Sad)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.94,\n                "join_silence_ms": 155,\n                "trailing_silence_ms": 310,\n                "num_step": 32,\n                "guidance_scale": 4.15,\n            },\n            "gain_db": -0.45,\n        },\n\n        "Hào hứng (Excited)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 1.018,\n                "join_silence_ms": 116,\n                "trailing_silence_ms": 226,\n                "num_step": 32,\n                "guidance_scale": 3.9,\n            },\n            "gain_db": 0.28,\n        },\n\n        "Giận dữ (Angry)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 1.012,\n                "join_silence_ms": 118,\n                "trailing_silence_ms": 232,\n                "num_step": 32,\n                "guidance_scale": 4.05,\n            },\n            "gain_db": 0.24,\n        },\n\n        "Nhẹ nhàng (Gentle)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.955,\n                "join_silence_ms": 150,\n                "trailing_silence_ms": 300,\n                "num_step": 32,\n                "guidance_scale": 4.12,\n            },\n            "gain_db": -0.22,\n        },\n    },\n\n    "my": {\n        "Mặc định": {\n            "overrides": {},\n            "gain_db": 0.0,\n        },\n\n        "Vui vẻ (Happy)": {\n            "overrides": {\n                "pitch_shift": 1.01,\n                "speed": 1.035,\n                "join_silence_ms": 108,\n                "trailing_silence_ms": 198,\n            },\n            "gain_db": 0.42,\n        },\n\n        "Buồn bã (Sad)": {\n            "overrides": {\n                "pitch_shift": 0.99,\n                "speed": 0.92,\n                "join_silence_ms": 170,\n                "trailing_silence_ms": 440,\n            },\n            "gain_db": -1.0,\n        },\n\n        "Hào hứng (Excited)": {\n            "overrides": {\n                "pitch_shift": 1.015,\n                "speed": 1.06,\n                "join_silence_ms": 88,\n                "trailing_silence_ms": 165,\n            },\n            "gain_db": 0.65,\n        },\n\n        "Giận dữ (Angry)": {\n            "overrides": {\n                "pitch_shift": 0.99,\n                "speed": 1.05,\n                "join_silence_ms": 82,\n                "trailing_silence_ms": 160,\n            },\n            "gain_db": 0.62,\n        },\n\n        "Nhẹ nhàng (Gentle)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.94,\n                "join_silence_ms": 145,\n                "trailing_silence_ms": 315,\n            },\n            "gain_db": -0.45,\n        },\n    },\n\n    "km": {\n        "Mặc định": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.975,\n                "join_silence_ms": 142,\n                "trailing_silence_ms": 286,\n                "num_step": 36,\n                "guidance_scale": 4.12,\n                "max_segment_chars": 58,\n            },\n            "gain_db": 0.0,\n        },\n\n        "Vui vẻ (Happy)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.99,\n                "join_silence_ms": 134,\n                "trailing_silence_ms": 262,\n                "num_step": 36,\n                "guidance_scale": 4.02,\n                "max_segment_chars": 56,\n            },\n            "gain_db": 0.16,\n        },\n\n        "Ngạc nhiên (Surprised)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.995,\n                "join_silence_ms": 128,\n                "trailing_silence_ms": 252,\n                "num_step": 36,\n                "guidance_scale": 4.14,\n                "max_segment_chars": 54,\n            },\n            "gain_db": 0.18,\n        },\n\n        "Buồn bã (Sad)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.935,\n                "join_silence_ms": 158,\n                "trailing_silence_ms": 320,\n                "num_step": 36,\n                "guidance_scale": 4.22,\n                "max_segment_chars": 60,\n            },\n            "gain_db": -0.38,\n        },\n\n        "Hào hứng (Excited)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 1.0,\n                "join_silence_ms": 124,\n                "trailing_silence_ms": 242,\n                "num_step": 36,\n                "guidance_scale": 3.98,\n                "max_segment_chars": 54,\n            },\n            "gain_db": 0.24,\n        },\n\n        "Giận dữ (Angry)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.998,\n                "join_silence_ms": 122,\n                "trailing_silence_ms": 238,\n                "num_step": 36,\n                "guidance_scale": 4.12,\n                "max_segment_chars": 54,\n            },\n            "gain_db": 0.22,\n        },\n\n        "Nhẹ nhàng (Gentle)": {\n            "overrides": {\n                "pitch_shift": 1.0,\n                "speed": 0.955,\n                "join_silence_ms": 152,\n                "trailing_silence_ms": 304,\n                "num_step": 36,\n                "guidance_scale": 4.18,\n                "max_segment_chars": 60,\n            },\n            "gain_db": -0.2,\n        },\n    },\n}\n\n\nAD_PRESETS_BY_LANG: Dict[str, Dict[str, Dict[str, Any]]] = {\n    "vi": {\n        "Không bổ trợ": {\n            "overrides": {},\n            "gain_db": 0.0,\n        },\n\n        "Cường điệu rất nhẹ": {\n            "overrides": {\n                "speed": 1.015,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 105,\n                "trailing_silence_ms": 205,\n            },\n            "gain_db": 0.18,\n        },\n\n        "Cường điệu nhẹ": {\n            "overrides": {\n                "speed": 1.035,\n                "pitch_shift": 1.005,\n                "join_silence_ms": 92,\n                "trailing_silence_ms": 180,\n            },\n            "gain_db": 0.38,\n        },\n\n        "Cường điệu vừa": {\n            "overrides": {\n                "speed": 1.055,\n                "pitch_shift": 1.01,\n                "join_silence_ms": 82,\n                "trailing_silence_ms": 160,\n            },\n            "gain_db": 0.62,\n        },\n\n        "Cường điệu mạnh": {\n            "overrides": {\n                "speed": 1.075,\n                "pitch_shift": 1.012,\n                "join_silence_ms": 72,\n                "trailing_silence_ms": 145,\n            },\n            "gain_db": 0.85,\n        },\n    },\n\n    "lo": {\n        "Không bổ trợ": {\n            "overrides": {},\n            "gain_db": 0.0,\n        },\n\n        "Cường điệu rất nhẹ": {\n            "overrides": {\n                "speed": 0.995,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 136,\n                "trailing_silence_ms": 268,\n                "num_step": 32,\n                "guidance_scale": 4.0,\n            },\n            "gain_db": 0.06,\n        },\n\n        "Cường điệu nhẹ": {\n            "overrides": {\n                "speed": 1.0,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 132,\n                "trailing_silence_ms": 260,\n                "num_step": 32,\n                "guidance_scale": 3.96,\n            },\n            "gain_db": 0.12,\n        },\n\n        "Cường điệu vừa": {\n            "overrides": {\n                "speed": 1.008,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 126,\n                "trailing_silence_ms": 248,\n                "num_step": 32,\n                "guidance_scale": 3.92,\n            },\n            "gain_db": 0.2,\n        },\n\n        "Cường điệu mạnh": {\n            "overrides": {\n                "speed": 1.015,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 120,\n                "trailing_silence_ms": 238,\n                "num_step": 32,\n                "guidance_scale": 3.88,\n            },\n            "gain_db": 0.28,\n        },\n    },\n\n    "my": {\n        "Không bổ trợ": {\n            "overrides": {},\n            "gain_db": 0.0,\n        },\n\n        "Cường điệu rất nhẹ": {\n            "overrides": {\n                "speed": 1.01,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 112,\n                "trailing_silence_ms": 215,\n            },\n            "gain_db": 0.14,\n        },\n\n        "Cường điệu nhẹ": {\n            "overrides": {\n                "speed": 1.03,\n                "pitch_shift": 1.005,\n                "join_silence_ms": 98,\n                "trailing_silence_ms": 188,\n            },\n            "gain_db": 0.32,\n        },\n\n        "Cường điệu vừa": {\n            "overrides": {\n                "speed": 1.05,\n                "pitch_shift": 1.01,\n                "join_silence_ms": 84,\n                "trailing_silence_ms": 164,\n            },\n            "gain_db": 0.52,\n        },\n\n        "Cường điệu mạnh": {\n            "overrides": {\n                "speed": 1.065,\n                "pitch_shift": 1.012,\n                "join_silence_ms": 76,\n                "trailing_silence_ms": 150,\n            },\n            "gain_db": 0.72,\n        },\n    },\n\n    "km": {\n        "Không bổ trợ": {\n            "overrides": {},\n            "gain_db": 0.0,\n        },\n\n        "Cường điệu rất nhẹ": {\n            "overrides": {\n                "speed": 0.985,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 138,\n                "trailing_silence_ms": 272,\n                "num_step": 36,\n                "guidance_scale": 4.12,\n                "max_segment_chars": 58,\n            },\n            "gain_db": 0.04,\n        },\n\n        "Cường điệu nhẹ": {\n            "overrides": {\n                "speed": 0.992,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 132,\n                "trailing_silence_ms": 260,\n                "num_step": 36,\n                "guidance_scale": 4.08,\n                "max_segment_chars": 56,\n            },\n            "gain_db": 0.1,\n        },\n\n        "Cường điệu vừa": {\n            "overrides": {\n                "speed": 1.0,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 126,\n                "trailing_silence_ms": 248,\n                "num_step": 36,\n                "guidance_scale": 4.02,\n                "max_segment_chars": 54,\n            },\n            "gain_db": 0.16,\n        },\n\n        "Cường điệu mạnh": {\n            "overrides": {\n                "speed": 1.006,\n                "pitch_shift": 1.0,\n                "join_silence_ms": 120,\n                "trailing_silence_ms": 238,\n                "num_step": 36,\n                "guidance_scale": 3.98,\n                "max_segment_chars": 52,\n            },\n            "gain_db": 0.24,\n        },\n    },\n}\n\nLANGUAGE_ALIASES: Dict[str, str] = {\n    "vi": "vi",\n    "vn": "vi",\n    "lo": "lo",\n    "la": "lo",\n    "km": "km",\n    "kh": "km",\n    "khmer": "km",\n    "th": "th",\n    "thai": "th",\n    "my": "my",\n    "mm": "my",\n    "myanmar": "my",\n}\n\nPROMPT_TEXT_FILES: Dict[str, tuple[str, str]] = {\n    "vi": ("vietnam_prompt_voice", "vietnam_prompt.txt"),\n    "km": ("khmer_prompt_voice", "khmer_prompt.txt"),\n    "my": ("myanmar_prompt_voice", "myanmar_prompt.txt"),\n}\n\nVOICE_DEFINITIONS: Dict[str, List[Dict[str, Any]]] = {\n    "vi": [\n        {"preset_key": "vi_nam_ke_chuyen", "label": "Giọng nam kể chuyện", "folder": "vietnam_prompt_voice", "audio_file": "giong_nam_ke_chuyen.mp3", "style_tags": ["storytelling", "warm", "soft"], "aliases": ["giong_nam_ke_chuyen"]},\n        {"preset_key": "vi_nam_qc", "label": "Giọng nam quảng cáo", "folder": "vietnam_prompt_voice", "audio_file": "giong_nam_qc.mp3", "style_tags": ["advertising", "bright", "decisive"], "aliases": ["giong_nam_qc"]},\n        {"preset_key": "vi_nam_truong_thanh", "label": "Giọng nam trưởng thành", "folder": "vietnam_prompt_voice", "audio_file": "giong_nam_truong_thanh.mp3", "style_tags": ["adult", "stable", "balanced"], "aliases": ["giong_nam_truong_thanh"]},\n        {"preset_key": "vi_nu_ke_chuyen", "label": "Giọng nữ kể chuyện", "folder": "vietnam_prompt_voice", "audio_file": "giong_nu_ke_chuyen.mp3", "style_tags": ["storytelling", "female", "soft"], "aliases": ["giong_nu_ke_chuyen"]},\n        {"preset_key": "vi_nu_qc", "label": "Giọng nữ quảng cáo", "folder": "vietnam_prompt_voice", "audio_file": "giong_nu_qc.mp3", "style_tags": ["advertising", "female", "clear"], "aliases": ["giong_nu_qc"]},\n        {"preset_key": "vi_tre_em_qc", "label": "Giọng trẻ em quảng cáo", "folder": "vietnam_prompt_voice", "audio_file": "giong_tre_em_qc.wav", "style_tags": ["advertising", "child", "playful"], "aliases": ["giong_tre_em_qc"]},\n    ],\n    "lo": [],\n    "km": [\n        {"preset_key": "km_1", "label": "Khmer 1", "folder": "khmer_prompt_voice", "audio_file": "1_khmer_audio_prompt.wav", "style_tags": ["neutral", "general"], "aliases": ["1_khmer_audio_prompt"]},\n        {"preset_key": "km_2", "label": "Khmer 2", "folder": "khmer_prompt_voice", "audio_file": "2_khmer_audio_prompt.wav", "style_tags": ["gentle", "natural"], "aliases": ["2_khmer_audio_prompt"]},\n        {"preset_key": "km_3", "label": "Khmer 3", "folder": "khmer_prompt_voice", "audio_file": "3_khmer_audio_prompt.wav", "style_tags": ["advertising", "energetic"], "aliases": ["3_khmer_audio_prompt"]},\n        {"preset_key": "km_4", "label": "Khmer 4", "folder": "khmer_prompt_voice", "audio_file": "4_khmer_audio_prompt.wav", "style_tags": ["stable", "serious"], "aliases": ["4_khmer_audio_prompt"]},\n    ],\n    "th": [],\n    "my": [\n        {"preset_key": "my_1", "label": "Myanmar 1", "folder": "myanmar_prompt_voice", "audio_file": "1_myanmar_audio_prompt.wav", "style_tags": ["neutral"], "aliases": ["1_myanmar_audio_prompt"]},\n        {"preset_key": "my_2", "label": "Myanmar 2", "folder": "myanmar_prompt_voice", "audio_file": "2_myanmar_audio_prompt.wav", "style_tags": ["gentle"], "aliases": ["2_myanmar_audio_prompt"]},\n        {"preset_key": "my_3", "label": "Myanmar 3", "folder": "myanmar_prompt_voice", "audio_file": "3_myanmar_audio_prompt.wav", "style_tags": ["advertising", "energetic"], "aliases": ["3_myanmar_audio_prompt"]},\n        {"preset_key": "my_4", "label": "Myanmar 4", "folder": "myanmar_prompt_voice", "audio_file": "4_myanmar_audio_prompt.wav", "style_tags": ["stable", "balanced"], "aliases": ["4_myanmar_audio_prompt"]},\n    ],\n}\n\n\ndef canonical_lang(lang: Optional[str]) -> str:\n    key = str(lang or "").strip().lower()\n    return LANGUAGE_ALIASES.get(key, key)\n\n\ndef _looks_like_prompt_root(path: Path) -> bool:\n    expected = ("vietnam_prompt_voice", "khmer_prompt_voice", "myanmar_prompt_voice")\n    return path.exists() and path.is_dir() and any((path / name).exists() for name in expected)\n\n\ndef _find_prompt_root() -> Optional[Path]:\n    env_root = os.getenv("PROMPT_VOICE_ROOT")\n    direct_candidates = [\n        Path(env_root).expanduser() if env_root else None,\n        Path("/runpod-volume/prompt_voices"),\n        Path("/app/prompt_voices"),\n        Path(__file__).resolve().parents[1] / "prompt_voices",\n        Path.cwd() / "prompt_voices",\n    ]\n    for candidate in direct_candidates:\n        if candidate and _looks_like_prompt_root(candidate):\n            return candidate.resolve()\n    return None\n\n\nPROMPT_VOICE_ROOT = _find_prompt_root()\n\n\ndef _read_text_if_exists(path: Path) -> str:\n    if not path.exists():\n        return ""\n    try:\n        return path.read_text(encoding="utf-8").strip()\n    except Exception:\n        return ""\n\n\ndef _prompt_text_path(lang: str) -> Optional[Path]:\n    lang = canonical_lang(lang)\n    if not PROMPT_VOICE_ROOT or lang not in PROMPT_TEXT_FILES:\n        return None\n    folder, filename = PROMPT_TEXT_FILES[lang]\n    return PROMPT_VOICE_ROOT / folder / filename\n\n\ndef _prompt_text(lang: str) -> str:\n    path = _prompt_text_path(lang)\n    return _read_text_if_exists(path) if path else ""\n\n\ndef get_emotion_presets_for_lang(lang: str) -> Dict[str, Any]:\n    lang = canonical_lang(lang)\n    return EMOTION_PRESETS_BY_LANG.get(lang, EMOTION_PRESETS_BY_LANG.get("vi", {}))\n\n\ndef get_ad_presets_for_lang(lang: str) -> Dict[str, Any]:\n    lang = canonical_lang(lang)\n    return AD_PRESETS_BY_LANG.get(lang, {"Không bổ trợ": {"overrides": {}, "gain_db": 0.0}})\n\n\ndef _normalize_style_key(value: Optional[str]) -> str:\n    text = unicodedata.normalize("NFD", str(value or ""))\n    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")\n    return " ".join(text.lower().strip().split())\n\n\ndef _resolve_style_key(options: Dict[str, Any], key: Optional[str]) -> Optional[str]:\n    if key in options:\n        return str(key)\n    normalized = _normalize_style_key(key)\n    for candidate in options:\n        if _normalize_style_key(candidate) == normalized:\n            return candidate\n    return None\n\n\ndef get_lang_config(lang: str, overrides: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:\n    lang = canonical_lang(lang)\n    cfg = dict(BASE_OMNI_CONFIG)\n    cfg.update(LANGUAGE_PRESETS.get(lang, {}))\n    cfg.update(DEPLOY_LANGUAGE_CONFIGS.get(lang, {}))\n    if overrides:\n        cfg.update({key: value for key, value in overrides.items() if value is not None})\n    return cfg\n\n\ndef get_effective_config(\n    lang: str,\n    emotion_key: str,\n    ad_key: str = "Không bổ trợ",\n    manual_overrides: Optional[Dict[str, Any]] = None,\n) -> Dict[str, Any]:\n    lang = canonical_lang(lang)\n    cfg = get_lang_config(lang)\n    ad_options = get_ad_presets_for_lang(lang)\n    emotion_options = get_emotion_presets_for_lang(lang)\n    resolved_ad_key = _resolve_style_key(ad_options, ad_key)\n    resolved_emotion_key = _resolve_style_key(emotion_options, emotion_key)\n    ad_cfg = ad_options.get(resolved_ad_key or "", {"overrides": {}, "gain_db": 0.0})\n    emotion_cfg = emotion_options.get(resolved_emotion_key or "", {"overrides": {}, "gain_db": 0.0})\n    \n    # Combine overrides: AD adjustments are baseline, Emotion adjustments are additive relative to 1.0\n    # This prevents Emotion from completely wiping out AD intensity changes.\n    ad_overrides = ad_cfg.get("overrides", {})\n    emotion_overrides = emotion_cfg.get("overrides", {})\n\n    cfg.update(ad_overrides)\n    for key, val in emotion_overrides.items():\n        if key in ["speed", "pitch_shift"] and key in cfg:\n            # Additive relative to 1.0: final = current_adjusted + (emotion_val - 1.0)\n            cfg[key] = float(cfg[key]) + (float(val) - 1.0)\n        else:\n            # Other parameters (silences, num_step) still use emotion-specific values\n            cfg[key] = val\n\n    if manual_overrides:\n        cfg.update({key: value for key, value in manual_overrides.items() if value is not None})\n    return cfg\n\n\ndef safe_gain_for_style(lang: str, emotion_key: str, ad_key: str, enabled: bool = True) -> float:\n    if not enabled:\n        return 0.0\n    lang = canonical_lang(lang)\n    emotion_options = get_emotion_presets_for_lang(lang)\n    ad_options = get_ad_presets_for_lang(lang)\n    emotion_cfg = emotion_options.get(_resolve_style_key(emotion_options, emotion_key) or "", {"gain_db": 0.0})\n    ad_cfg = ad_options.get(_resolve_style_key(ad_options, ad_key) or "", {"gain_db": 0.0})\n    return float(emotion_cfg.get("gain_db", 0.0)) + float(ad_cfg.get("gain_db", 0.0))\n\n\ndef apply_ad_safe_guard(lang: str, cfg: Dict[str, Any], enabled: bool = True) -> Dict[str, Any]:\n    return dict(cfg)\n\n\ndef clamp_prosody(lang: str, cfg: Dict[str, Any]) -> Dict[str, Any]:\n    lang = canonical_lang(lang)\n    clamped = dict(cfg)\n    if lang == "vi":\n        speed_min, speed_max = 0.88, 1.18\n        pitch_min, pitch_max = 0.95, 1.08\n    elif lang == "my":\n        speed_min, speed_max = 0.90, 2.0\n        pitch_min, pitch_max = 0.96, 1.06\n    else:\n        speed_min, speed_max = 0.88, 1.15\n        pitch_min, pitch_max = 0.94, 1.08\n    clamped["speed"] = max(speed_min, min(speed_max, float(clamped.get("speed", 1.0))))\n    clamped["pitch_shift"] = max(pitch_min, min(pitch_max, float(clamped.get("pitch_shift", 1.0))))\n    return clamped\n\n\ndef stabilize_clone_config(lang: str, cfg: Dict[str, Any]) -> Dict[str, Any]:\n    return dict(cfg)\n\n\ndef _build_voice_preset(defn: Dict[str, Any], lang: str) -> Dict[str, Any]:\n    audio_path = None\n    if PROMPT_VOICE_ROOT:\n        audio_path = PROMPT_VOICE_ROOT / str(defn["folder"]) / str(defn["audio_file"])\n    ref_text_path = _prompt_text_path(lang)\n    ref_text = _prompt_text(lang)\n    return {\n        "preset_key": str(defn["preset_key"]),\n        "label": str(defn["label"]),\n        "language": canonical_lang(lang),\n        "audio_path": str(audio_path) if audio_path else "",\n        "exists": bool(audio_path and audio_path.exists()),\n        "reference_text_path": str(ref_text_path) if ref_text_path else "",\n        "reference_text": ref_text,\n        "reference_text_exists": bool(ref_text_path and ref_text_path.exists() and ref_text),\n        "style_tags": list(defn.get("style_tags", [])),\n        "aliases": list(defn.get("aliases", [])),\n    }\n\n\ndef list_voice_presets() -> List[Dict[str, Any]]:\n    items: List[Dict[str, Any]] = []\n    for lang, definitions in VOICE_DEFINITIONS.items():\n        for definition in definitions:\n            items.append(_build_voice_preset(definition, lang))\n    return items\n\n\ndef get_voice_preset(preset_key: Optional[str]) -> Optional[Dict[str, Any]]:\n    if not preset_key:\n        return None\n    normalized = str(preset_key).strip().lower()\n    for preset in list_voice_presets():\n        keys = [str(preset.get("preset_key", "")).lower()] + [str(alias).lower() for alias in preset.get("aliases", [])]\n        if normalized in keys:\n            return preset\n    return None\n\n\ndef build_prompt_voice_healthcheck() -> Dict[str, Any]:\n    presets = list_voice_presets()\n    available = [item for item in presets if item.get("exists")]\n    missing = [item for item in presets if not item.get("exists")]\n    reference_text_paths = {\n        lang: str(path) if path else None\n        for lang, path in ((lang, _prompt_text_path(lang)) for lang in LANGUAGE_LABELS)\n    }\n    reference_text_exists = {\n        lang: bool(path and Path(path).exists())\n        for lang, path in reference_text_paths.items()\n    }\n    return {\n        "prompt_voice_root": str(PROMPT_VOICE_ROOT) if PROMPT_VOICE_ROOT else None,\n        "available_presets": available,\n        "missing_presets": missing,\n        "available_count": len(available),\n        "missing_count": len(missing),\n        "reference_text_paths": reference_text_paths,\n        "reference_text_exists": reference_text_exists,\n    }\n',
    'text_normalization.py': 'from __future__ import annotations\n\nimport re\nimport unicodedata\nfrom typing import Any, Dict, List, Optional\n\nfrom .presets import canonical_lang\n\ndef add_config_text_omni(text: str) -> str:\n    return f" . {text.strip()} . "\n\n\ndef _replace_common_symbols(text: str) -> str:\n    replacements = {\n        "\\u00a0": " ",\n        "\\u200b": " ",\n        "\\ufeff": " ",\n        "“": \'"\',\n        "”": \'"\',\n        "‘": "\'",\n        "’": "\'",\n        "–": "-",\n        "—": "-",\n        "•": ". ",\n        "·": ". ",\n    }\n    for src, dest in replacements.items():\n        text = text.replace(src, dest)\n    return text\n\n\ndef _normalize_list_breaks(text: str) -> str:\n    text = re.sub(r"\\s*[\\r\\n]+\\s*", ". ", text)\n    text = re.sub(r"\\s*[|]\\s*", " / ", text)\n    return text\n\n\ndef _collapse_punctuation_runs(text: str, keep_chars: str) -> str:\n    pattern = rf"([{re.escape(keep_chars)}])\\1+"\n    return re.sub(pattern, r"\\1", text)\n\n\ndef _protect_numeric_punctuation(text: str) -> str:\n    replacements = {\n        ".": "__NUM_DOT__",\n        ",": "__NUM_COMMA__",\n        ":": "__NUM_COLON__",\n        "/": "__NUM_SLASH__",\n    }\n    for punct, marker in replacements.items():\n        text = re.sub(rf"(?<=\\d){re.escape(punct)}(?=\\d)", marker, text)\n    return text\n\n\ndef _restore_numeric_punctuation(text: str) -> str:\n    return (\n        text.replace("__NUM_DOT__", ".")\n        .replace("__NUM_COMMA__", ",")\n        .replace("__NUM_COLON__", ":")\n        .replace("__NUM_SLASH__", "/")\n    )\n\n\ndef _basic_text_cleanup(text: str) -> str:\n    text = unicodedata.normalize("NFC", str(text or ""))\n    text = _replace_common_symbols(text)\n    text = _normalize_list_breaks(text)\n    text = text.replace("\\r", " ").replace("\\n", " ")\n    text = re.sub(r"\\s+", " ", text).strip()\n    return text\n\n\ndef _normalize_number_separators(text: str) -> str:\n    text = re.sub(r"(?<=\\d)\\s*,\\s*(?=\\d{3}\\b)", ",", text)\n    text = re.sub(r"(?<=\\d)\\s*\\.\\s*(?=\\d)", ".", text)\n    text = re.sub(r"(?<=\\d)\\s*:\\s*(?=\\d)", ":", text)\n    text = re.sub(r"(?<=\\d)\\s*/\\s*(?=\\d)", "/", text)\n    return text\n\n\ndef _trim_punctuation_spacing(text: str, punctuation: str, protect_numeric: bool = False) -> str:\n    text = re.sub(rf"\\s+([{re.escape(punctuation)}])", r"\\1", text)\n    if protect_numeric:\n        leading_safe = "".join(ch for ch in punctuation if ch in ",;!?")\n        numeric_sensitive = "".join(ch for ch in punctuation if ch in ".:/")\n        other_chars = "".join(ch for ch in punctuation if ch not in f"{leading_safe}{numeric_sensitive}")\n        if leading_safe:\n            text = re.sub(rf"([{re.escape(leading_safe)}])(\\S)", r"\\1 \\2", text)\n        if numeric_sensitive:\n            text = re.sub(rf"([{re.escape(numeric_sensitive)}])(?!\\d)(\\S)", r"\\1 \\2", text)\n        if other_chars:\n            text = re.sub(rf"([{re.escape(other_chars)}])(\\S)", r"\\1 \\2", text)\n    else:\n        text = re.sub(rf"([{re.escape(punctuation)}])(\\S)", r"\\1 \\2", text)\n    return re.sub(r"\\s+", " ", text).strip()\n\n\ndef normalize_vietnamese_tts_text(text: str) -> str:\n    text = re.sub(r"(\\d+)\\s*%", r"\\1 phần trăm", text)\n    text = re.sub(r"(\\d+)\\s*(?:°c|ºc)", r"\\1 độ C", text, flags=re.IGNORECASE)\n    text = re.sub(r"(\\d+)\\s*(?:km/h|kmh)", r"\\1 ki lô mét trên giờ", text, flags=re.IGNORECASE)\n    text = re.sub(r"(\\d+)\\s*(?:vnd|đ|₫)", r"\\1 đồng", text, flags=re.IGNORECASE)\n    text = re.sub(r"\\bTP\\.\\s*HCM\\b", "Thành phố Hồ Chí Minh", text, flags=re.IGNORECASE)\n    text = re.sub(r"\\bQ\\.\\s*(\\d+)\\b", r"quận \\1", text, flags=re.IGNORECASE)\n    text = _normalize_number_separators(text)\n    text = re.sub(r"(?<=\\w)\\s*-\\s*(?=\\w)", "-", text)\n    text = _collapse_punctuation_runs(text, ",.;:!?")\n    return _trim_punctuation_spacing(text, ",.;:!?", protect_numeric=True)\n\ndef normalize_myanmar_tts_text(text: str) -> str:\n    text = _normalize_number_separators(text)\n    text = re.sub(r"(?<=\\w)\\s*-\\s*(?=\\w)", "-", text)\n    text = re.sub(r"\\s*[,;]\\s*", "၊ ", text)\n    text = re.sub(r"\\s*[.!?]\\s*", "။ ", text)\n    text = _collapse_punctuation_runs(text, "၊။")\n    return _trim_punctuation_spacing(text, "၊။")\n\ndef normalize_khmer_tts_text(text: str) -> str:\n    text = re.sub(r"(?<=\\d)\\s*%\\s*", " ភាគរយ", text)\n    text = _normalize_number_separators(text)\n    text = re.sub(r"\\s*[.!?]\\s*", "។ ", text)\n    text = _collapse_punctuation_runs(text, "។៕,;:")\n    return _trim_punctuation_spacing(text, "។៕,;:/", protect_numeric=True)\n\ndef preprocess_text_for_tts(text: str, lang: str, is_reference: bool = False) -> str:\n    lang = canonical_lang(lang)\n    text = _basic_text_cleanup(text)\n    if not text:\n        return text\n\n    if is_reference:\n        return text\n\n    text = text.replace("…", ".")\n    text = _collapse_punctuation_runs(text, ",.;:!?/|")\n\n    if lang == "vi":\n        return normalize_vietnamese_tts_text(text)\n    if lang == "my":\n        return normalize_myanmar_tts_text(text)\n    if lang == "km":\n        return normalize_khmer_tts_text(text)\n    if lang in {"lo", "th"}:\n        text = _trim_punctuation_spacing(text, ",.;:!?")\n\n    return text\n\ndef clean_and_segment_text(text: str) -> List[str]:\n    text = _protect_numeric_punctuation(re.sub(r"\\s+", " ", str(text or "")).strip())\n    punct_pattern = r"(\\.\\.+|…+|[.!?,;:/—–\\-，。？！、；៖၊။៕])"\n    parts = re.split(punct_pattern, text)\n    segments: List[str] = []\n    current_text = ""\n    for part in parts:\n        if not part:\n            continue\n        if re.fullmatch(punct_pattern, part):\n            if current_text.strip():\n                segments.append((current_text + part).strip())\n                current_text = ""\n        else:\n            current_text += part\n    if current_text.strip():\n        segments.append(current_text.strip())\n    return [_restore_numeric_punctuation(seg) for seg in segments if seg.strip()]\n\ndef _split_segment_to_max_chars(segment: str, max_chars: Optional[int]) -> List[str]:\n    segment = _protect_numeric_punctuation((segment or "").strip())\n    if not segment:\n        return []\n    if not max_chars or len(segment) <= max_chars:\n        return [_restore_numeric_punctuation(segment)]\n\n    sep_pattern = r"([,;:，、；៖၊。។៕])"\n    tokens = re.split(sep_pattern, segment)\n    parts: List[str] = []\n    current = ""\n    for token in tokens:\n        if not token:\n            continue\n        candidate = f"{current}{token}".strip()\n        if current and len(candidate) > max_chars:\n            parts.append(current.strip())\n            current = token.strip()\n        else:\n            current = candidate\n    if current.strip():\n        parts.append(current.strip())\n\n    refined: List[str] = []\n    for part in parts:\n        if len(part) <= max_chars:\n            refined.append(part)\n            continue\n        words = part.split(" ")\n        current_word_chunk = ""\n        for word in words:\n            candidate = f"{current_word_chunk} {word}".strip()\n            if current_word_chunk and len(candidate) > max_chars:\n                refined.append(current_word_chunk.strip())\n                current_word_chunk = word\n            else:\n                current_word_chunk = candidate\n        if current_word_chunk.strip():\n            refined.append(current_word_chunk.strip())\n\n    final_parts: List[str] = []\n    for part in refined:\n        if len(part) <= max_chars:\n            final_parts.append(part)\n            continue\n        start = 0\n        while start < len(part):\n            final_parts.append(part[start:start + max_chars].strip())\n            start += max_chars\n    return [_restore_numeric_punctuation(part) for part in final_parts if part]\n\ndef _merge_short_chunks(\n    chunks: List[Dict[str, Any]],\n    max_chars: Optional[int],\n    min_chars: Optional[int],\n) -> List[Dict[str, Any]]:\n    if not chunks or not max_chars or not min_chars:\n        return chunks\n\n    merged: List[Dict[str, Any]] = []\n    idx = 0\n    while idx < len(chunks):\n        current = dict(chunks[idx])\n        current_text = str(current.get("text", "")).strip()\n        if not current_text:\n            idx += 1\n            continue\n\n        current["text"] = current_text\n        current_len = len(current_text)\n        if current_len >= min_chars:\n            merged.append(current)\n            idx += 1\n            continue\n\n        is_terminal_short_chunk = current_text.endswith((".", "!", "?", "။", "。", "؟", "！", "？"))\n\n        if idx + 1 < len(chunks):\n            next_chunk = dict(chunks[idx + 1])\n            next_text = str(next_chunk.get("text", "")).strip()\n            if next_text:\n                combined = f"{current_text} {next_text}".strip()\n                if len(combined) <= max_chars:\n                    next_chunk["text"] = combined\n                    next_chunk["pause_ms"] = int(next_chunk.get("pause_ms", current.get("pause_ms", 0)))\n                    chunks[idx + 1] = next_chunk\n                    idx += 1\n                    continue\n\n        if merged and not is_terminal_short_chunk:\n            prev = dict(merged[-1])\n            combined = f"{prev[\'text\']} {current_text}".strip()\n            if len(combined) <= max_chars:\n                prev["text"] = combined\n                prev["pause_ms"] = int(current.get("pause_ms", prev.get("pause_ms", 0)))\n                merged[-1] = prev\n                idx += 1\n                continue\n\n        merged.append(current)\n        idx += 1\n\n    return merged\n\ndef segment_text_with_pauses(\n    text: str,\n    join_silence_ms: int,\n    max_chars: Optional[int] = None,\n    min_chars: Optional[int] = None,\n    lang: Optional[str] = None,\n) -> List[Dict[str, Any]]:\n    pause_scale = {\n        ",": 0.75,\n        ";": 0.95,\n        ":": 1.0,\n        ".": 1.15,\n        "!": 1.2,\n        "?": 1.2,\n        "…": 1.3,\n        "。": 1.15,\n        "，": 0.75,\n        "？": 1.2,\n        "！": 1.2,\n        "、": 0.75,\n        "；": 0.95,\n        "៖": 1.0,\n    }\n    if canonical_lang(lang) == "my":\n        pause_scale["၊"] = 0.66\n        pause_scale["။"] = 1.00\n    segments = clean_and_segment_text(text)\n    result: List[Dict[str, Any]] = []\n    base_pause = max(40, int(join_silence_ms))\n    for segment in segments:\n        split_segments = _split_segment_to_max_chars(segment, max_chars)\n        for idx, piece in enumerate(split_segments):\n            ending = piece[-1] if piece else ""\n            pause_ms = int(base_pause * pause_scale.get(ending, 0.9 if idx < len(split_segments) - 1 else 1.0))\n            result.append({"text": piece, "pause_ms": pause_ms})\n    result = _merge_short_chunks(result, max_chars=max_chars, min_chars=min_chars)\n    if not result and text.strip():\n        result.append({"text": text.strip(), "pause_ms": base_pause})\n    return result\n\ndef reference_quality_note(lang: str, ref_text: Optional[str]) -> str:\n    ref_text = _basic_text_cleanup(ref_text or "")\n    lang = canonical_lang(lang)\n    if not ref_text:\n        return "missing_ref_text"\n    size = len(ref_text)\n    if lang == "vi":\n        if size < 20:\n            return "ref_text_too_short_vi"\n        if size > 220:\n            return "ref_text_too_long_vi"\n        return "ok_vi"\n    if lang == "my":\n        if size < 15:\n            return "ref_text_too_short_my"\n        if size > 180:\n            return "ref_text_too_long_my"\n        return "ok_my"\n    return "unchecked"\n\n\ndef build_ref_text(ref_text: Optional[str], fallback_text: Optional[str], lang: str) -> Optional[str]:\n    raw = (ref_text or "").strip()\n    if raw:\n        return preprocess_text_for_tts(raw, lang, is_reference=True)\n    fallback = (fallback_text or "").strip()\n    if fallback:\n        return preprocess_text_for_tts(fallback, lang, is_reference=True)\n    return None\n',
    'audio_processing.py': 'from __future__ import annotations\n\nimport base64\nimport binascii\nimport io\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional, Tuple\nfrom urllib.parse import urlparse\n\nimport numpy as np\nimport requests\nimport soundfile as sf\n\nDEFAULT_TARGET_SR = 24000\nEPS = 1e-8\n_LIBROSA_MODULE: Any = None\n_LIBROSA_IMPORT_ERROR: Optional[Exception] = None\n\n\ndef _get_librosa_module() -> Any:\n    global _LIBROSA_MODULE, _LIBROSA_IMPORT_ERROR\n    if _LIBROSA_MODULE is not None:\n        return _LIBROSA_MODULE\n    if _LIBROSA_IMPORT_ERROR is not None:\n        raise RuntimeError(f"librosa is unavailable in this environment: {_LIBROSA_IMPORT_ERROR}") from _LIBROSA_IMPORT_ERROR\n    try:\n        import librosa as librosa_module\n    except Exception as exc:\n        _LIBROSA_IMPORT_ERROR = exc\n        raise RuntimeError(f"librosa is unavailable in this environment: {exc}") from exc\n    _LIBROSA_MODULE = librosa_module\n    return _LIBROSA_MODULE\n\n\ndef ensure_dir(path: str | Path) -> Path:\n    p = Path(path)\n    p.mkdir(parents=True, exist_ok=True)\n    return p\n\n\ndef guess_extension_from_url(url: str, default: str = ".bin") -> str:\n    path = urlparse(url).path\n    suffix = Path(path).suffix\n    return suffix if suffix else default\n\n\ndef decode_base64_to_file(data: str, out_path: str | Path) -> Path:\n    payload = data.split(",", 1)[1] if data.startswith("data:") and "," in data else data\n    try:\n        raw = base64.b64decode(payload, validate=True)\n    except binascii.Error as exc:\n        raise ValueError("reference_audio_base64 is not valid base64") from exc\n    out = Path(out_path)\n    out.write_bytes(raw)\n    return out\n\n\ndef download_file(url: str, out_path: str | Path, timeout: int = 60) -> Path:\n    out = Path(out_path)\n    with requests.get(url, stream=True, timeout=timeout) as response:\n        response.raise_for_status()\n        with out.open("wb") as f:\n            for chunk in response.iter_content(chunk_size=1024 * 1024):\n                if chunk:\n                    f.write(chunk)\n    return out\n\n\ndef resolve_reference_audio_source(\n    *,\n    reference_audio_path: Optional[str],\n    reference_audio_url: Optional[str],\n    reference_audio_base64: Optional[str],\n    work_dir: str | Path,\n) -> Optional[Path]:\n    work_dir = ensure_dir(work_dir)\n\n    if reference_audio_path:\n        p = Path(reference_audio_path)\n        if not p.exists():\n            raise FileNotFoundError(f"reference_audio_path not found: {p}")\n        return p\n\n    if reference_audio_url:\n        suffix = guess_extension_from_url(reference_audio_url, default=".bin")\n        return download_file(reference_audio_url, work_dir / f"reference_from_url{suffix}")\n\n    if reference_audio_base64:\n        return decode_base64_to_file(reference_audio_base64, work_dir / "reference_from_base64.bin")\n\n    return None\n\n\ndef load_audio_mono(path: str | Path, sr: Optional[int] = None) -> Tuple[np.ndarray, int]:\n    librosa = _get_librosa_module()\n    audio, sample_rate = librosa.load(str(path), sr=sr, mono=True)\n    if audio.ndim != 1:\n        audio = np.mean(audio, axis=0)\n    audio = np.nan_to_num(audio).astype(np.float32)\n    return audio, int(sample_rate)\n\n\ndef _safe_trim(audio: np.ndarray, top_db: int = 35) -> Tuple[np.ndarray, bool]:\n    if audio.size == 0:\n        return audio, False\n    librosa = _get_librosa_module()\n    trimmed, idx = librosa.effects.trim(audio, top_db=top_db)\n    changed = bool(idx[0] > 0 or idx[1] < len(audio))\n    if trimmed.size == 0:\n        return audio, False\n    return trimmed.astype(np.float32), changed\n\n\ndef _remove_dc_offset(audio: np.ndarray) -> np.ndarray:\n    if audio.size == 0:\n        return audio.astype(np.float32)\n    return (audio - np.mean(audio)).astype(np.float32)\n\n\ndef _peak_normalize(audio: np.ndarray, target_peak: float = 0.95) -> np.ndarray:\n    if audio.size == 0:\n        return audio.astype(np.float32)\n    peak = float(np.max(np.abs(audio)))\n    if peak < EPS:\n        return audio.astype(np.float32)\n    if peak <= float(target_peak):\n        return audio.astype(np.float32)\n    return (audio / peak * float(target_peak)).astype(np.float32)\n\n\ndef _rms_dbfs(audio: np.ndarray) -> float:\n    if audio.size == 0:\n        return -120.0\n    rms = float(np.sqrt(np.mean(np.square(audio)) + EPS))\n    return 20.0 * np.log10(max(rms, EPS))\n\n\ndef _normalize_rms(audio: np.ndarray, target_dbfs: float = -22.0, min_gain: float = 0.6, max_gain: float = 2.5) -> np.ndarray:\n    if audio.size == 0:\n        return audio.astype(np.float32)\n    current = _rms_dbfs(audio)\n    gain = 10.0 ** ((target_dbfs - current) / 20.0)\n    gain = max(min_gain, min(max_gain, gain))\n    return (audio * gain).astype(np.float32)\n\n\ndef _limit_duration(audio: np.ndarray, sr: int, max_seconds: float) -> Tuple[np.ndarray, bool]:\n    max_samples = int(sr * max_seconds)\n    if max_samples <= 0 or audio.size <= max_samples:\n        return audio, False\n    return audio[:max_samples].astype(np.float32), True\n\n\ndef _estimate_snr_db(audio: np.ndarray, frame_length: int = 2048, hop_length: int = 512) -> float:\n    if audio.size < frame_length:\n        return 0.0\n    librosa = _get_librosa_module()\n    rms = librosa.feature.rms(y=audio, frame_length=frame_length, hop_length=hop_length, center=True).flatten()\n    if rms.size < 8:\n        return 0.0\n    signal = float(np.percentile(rms, 90))\n    noise = float(np.percentile(rms, 20))\n    if signal < EPS:\n        return 0.0\n    return float(20.0 * np.log10(max(signal, EPS) / max(noise, EPS)))\n\n\ndef _estimate_silence_ratio(audio: np.ndarray, frame_length: int = 2048, hop_length: int = 512) -> float:\n    if audio.size < frame_length:\n        return 0.0\n    librosa = _get_librosa_module()\n    intervals = librosa.effects.split(audio, top_db=35, frame_length=frame_length, hop_length=hop_length)\n    if len(intervals) == 0:\n        return 1.0\n    voiced = sum(max(0, end - start) for start, end in intervals)\n    return float(max(0.0, 1.0 - voiced / max(len(audio), 1)))\n\n\ndef analyze_reference_audio(audio: np.ndarray, sr: int) -> Dict[str, Any]:\n    duration_sec = float(len(audio) / max(sr, 1))\n    peak = float(np.max(np.abs(audio))) if audio.size else 0.0\n    clipping_ratio = float(np.mean(np.abs(audio) >= 0.995)) if audio.size else 0.0\n    rms_dbfs = _rms_dbfs(audio)\n    snr_db = _estimate_snr_db(audio)\n    silence_ratio = _estimate_silence_ratio(audio)\n    if audio.size:\n        librosa = _get_librosa_module()\n        zero_crossing_rate = float(np.mean(librosa.feature.zero_crossing_rate(y=audio)))\n    else:\n        zero_crossing_rate = 0.0\n\n    warnings = []\n    if duration_sec < 1.5:\n        warnings.append("reference_audio_short")\n    if clipping_ratio > 0.001:\n        warnings.append("reference_audio_possible_clipping")\n    if rms_dbfs < -36.0:\n        warnings.append("reference_audio_too_quiet")\n    if snr_db < 10.0:\n        warnings.append("reference_audio_low_snr")\n    if silence_ratio > 0.35:\n        warnings.append("reference_audio_contains_too_much_silence")\n\n    return {\n        "duration_sec": round(duration_sec, 4),\n        "peak": round(peak, 6),\n        "rms_dbfs": round(rms_dbfs, 3),\n        "estimated_snr_db": round(snr_db, 3),\n        "silence_ratio": round(silence_ratio, 4),\n        "clipping_ratio": round(clipping_ratio, 6),\n        "zero_crossing_rate": round(zero_crossing_rate, 6),\n        "warnings": warnings,\n    }\n\n\ndef _keep_voiced_regions(\n    audio: np.ndarray,\n    sr: int,\n    *,\n    top_db: int = 32,\n    frame_length: int = 2048,\n    hop_length: int = 512,\n    keep_silence_ms: int = 120,\n) -> Tuple[np.ndarray, bool]:\n    if audio.size == 0:\n        return audio.astype(np.float32), False\n\n    librosa = _get_librosa_module()\n    intervals = librosa.effects.split(audio, top_db=top_db, frame_length=frame_length, hop_length=hop_length)\n    if len(intervals) == 0:\n        return audio.astype(np.float32), False\n    if len(intervals) == 1 and intervals[0][0] == 0 and intervals[0][1] >= len(audio):\n        return audio.astype(np.float32), False\n\n    gap = np.zeros(int(sr * max(0, keep_silence_ms) / 1000.0), dtype=np.float32)\n    pieces = []\n    for start, end in intervals:\n        piece = audio[start:end].astype(np.float32)\n        if piece.size:\n            pieces.append(piece)\n    if not pieces:\n        return audio.astype(np.float32), False\n\n    if len(pieces) == 1:\n        rebuilt = pieces[0]\n    else:\n        joined_pieces = []\n        for idx, piece in enumerate(pieces):\n            if idx > 0 and gap.size:\n                joined_pieces.append(gap)\n            joined_pieces.append(piece)\n        rebuilt = np.concatenate(joined_pieces, axis=0)\n    return rebuilt.astype(np.float32), True\n\n\ndef _spectral_denoise(audio: np.ndarray, sr: int, strength: float = 0.18) -> Tuple[np.ndarray, bool]:\n    if audio.size < 2048 or strength <= 0:\n        return audio.astype(np.float32), False\n\n    librosa = _get_librosa_module()\n    n_fft = 1024\n    hop_length = 256\n    stft = librosa.stft(audio.astype(np.float32), n_fft=n_fft, hop_length=hop_length)\n    mag, phase = np.abs(stft), np.angle(stft)\n    if mag.size == 0:\n        return audio.astype(np.float32), False\n\n    frame_energy = np.mean(mag, axis=0)\n    if frame_energy.size < 8:\n        return audio.astype(np.float32), False\n\n    noise_threshold = np.percentile(frame_energy, 20)\n    noise_frames = mag[:, frame_energy <= noise_threshold]\n    if noise_frames.size == 0:\n        return audio.astype(np.float32), False\n\n    noise_profile = np.mean(noise_frames, axis=1, keepdims=True)\n    floor = 0.10\n    suppression = np.maximum(floor, 1.0 - float(strength) * noise_profile / np.maximum(mag, EPS))\n    clean_mag = mag * suppression\n    clean_stft = clean_mag * np.exp(1j * phase)\n    out = librosa.istft(clean_stft, hop_length=hop_length, length=len(audio))\n    return out.astype(np.float32), True\n\n\ndef preprocess_reference_audio(\n    source_path: str | Path,\n    output_path: str | Path,\n    *,\n    target_sr: int = DEFAULT_TARGET_SR,\n    trim_silence: bool = True,\n    trim_top_db: int = 35,\n    apply_vad: bool = True,\n    vad_top_db: int = 35,\n    max_internal_silence_ms: int = 120,\n    apply_denoise: bool = True,\n    denoise_strength: float = 0.18,\n    apply_rms_normalize: bool = True,\n    target_rms_dbfs: float = -22.0,\n    peak_normalize: bool = True,\n    peak_target: float = 0.92,\n    min_seconds: float = 1.5,\n    max_seconds: float = 10.0,\n) -> Dict[str, Any]:\n    raw_audio, original_sr = load_audio_mono(source_path, sr=None)\n    quality_before = analyze_reference_audio(raw_audio, original_sr)\n    duration_before = float(len(raw_audio) / max(original_sr, 1))\n\n    processed = raw_audio.astype(np.float32, copy=False)\n    if original_sr != target_sr:\n        librosa = _get_librosa_module()\n        processed = librosa.resample(processed, orig_sr=original_sr, target_sr=target_sr).astype(np.float32)\n\n    trimmed_silence_applied = False\n    vad_applied = False\n    denoise_applied = False\n    rms_applied = False\n    capped_to_max_duration = False\n    fell_back_to_resampled = False\n\n    processed = _remove_dc_offset(processed)\n\n    if trim_silence:\n        processed, trimmed_silence_applied = _safe_trim(processed, top_db=trim_top_db)\n\n    if apply_vad:\n        processed, vad_applied = _keep_voiced_regions(\n            processed,\n            target_sr,\n            top_db=vad_top_db,\n            keep_silence_ms=max_internal_silence_ms,\n        )\n\n    if apply_denoise:\n        processed, denoise_applied = _spectral_denoise(processed, target_sr, strength=denoise_strength)\n\n    processed = _remove_dc_offset(processed)\n\n    if apply_rms_normalize:\n        processed = _normalize_rms(processed, target_dbfs=target_rms_dbfs)\n        rms_applied = True\n\n    if peak_normalize:\n        processed = _peak_normalize(processed, target_peak=peak_target)\n\n    processed, capped_to_max_duration = _limit_duration(processed, target_sr, max_seconds=max_seconds)\n\n    duration_after = float(len(processed) / max(target_sr, 1))\n    if duration_after < float(min_seconds):\n        # Over-aggressive cleanup can destroy prompt quality for voice cloning.\n        # Fall back to a simpler resampled version instead of shipping a broken prompt.\n        processed = raw_audio.astype(np.float32, copy=False)\n        if original_sr != target_sr:\n            librosa = _get_librosa_module()\n            processed = librosa.resample(processed, orig_sr=original_sr, target_sr=target_sr).astype(np.float32)\n        processed = _remove_dc_offset(processed)\n        if apply_rms_normalize:\n            processed = _normalize_rms(processed, target_dbfs=target_rms_dbfs)\n            rms_applied = True\n        if peak_normalize:\n            processed = _peak_normalize(processed, target_peak=peak_target)\n        processed, capped_to_max_duration = _limit_duration(processed, target_sr, max_seconds=max_seconds)\n        fell_back_to_resampled = True\n\n    duration_after = float(len(processed) / max(target_sr, 1))\n    quality_after = analyze_reference_audio(processed, target_sr)\n    if fell_back_to_resampled:\n        quality_after["warnings"].append("preprocess_fallback_to_resampled_reference")\n\n    out = Path(output_path)\n    sf.write(str(out), processed, target_sr, subtype="PCM_16")\n\n    return {\n        "path": str(out),\n        "original_sample_rate": int(original_sr),\n        "sample_rate": int(target_sr),\n        "duration_before_sec": round(duration_before, 4),\n        "duration_after_sec": round(duration_after, 4),\n        "trimmed_silence": trimmed_silence_applied,\n        "applied_vad": vad_applied,\n        "applied_denoise": denoise_applied,\n        "applied_rms_normalize": rms_applied,\n        "capped_to_max_duration": capped_to_max_duration,\n        "fell_back_to_resampled_reference": fell_back_to_resampled,\n        "channels": 1,\n        "quality_before": quality_before,\n        "quality_after": quality_after,\n    }\n\n\ndef wav_bytes_from_array(audio: np.ndarray, sample_rate: int) -> bytes:\n    with io.BytesIO() as buf:\n        sf.write(buf, audio.astype(np.float32), sample_rate, format="WAV", subtype="PCM_16")\n        return buf.getvalue()\n\n\ndef encode_wav_base64(audio: np.ndarray, sample_rate: int) -> str:\n    return base64.b64encode(wav_bytes_from_array(audio, sample_rate)).decode("utf-8")\n\n\ndef write_output_wav(audio: np.ndarray, sample_rate: int, output_path: str | Path) -> Path:\n    output_path = Path(output_path)\n    ensure_dir(output_path.parent)\n    sf.write(str(output_path), audio.astype(np.float32), sample_rate, subtype="PCM_16")\n    return output_path\n',
    'engine.py': 'from __future__ import annotations\n\nimport gc\nimport glob\nimport hashlib\nimport inspect\nimport json\nimport math\nimport os\nimport tempfile\nimport time\nimport uuid\nfrom pathlib import Path\nfrom typing import Any, Dict, Optional, Tuple\n\nimport numpy as np\nimport soundfile as sf\n\nfrom .audio_processing import (\n    DEFAULT_TARGET_SR,\n    encode_wav_base64,\n    ensure_dir,\n    preprocess_reference_audio,\n    resolve_reference_audio_source,\n    write_output_wav,\n)\nfrom .presets import (\n    LANGUAGE_LABELS,\n    build_prompt_voice_healthcheck,\n    canonical_lang,\n    clamp_prosody,\n    get_effective_config,\n    safe_gain_for_style,\n    get_voice_preset,\n)\nfrom .text_normalization import (\n    add_config_text_omni,\n    build_ref_text,\n    preprocess_text_for_tts,\n    reference_quality_note,\n    segment_text_with_pauses,\n)\n\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\n\n_TORCH_MODULE: Any = None\n_TORCH_IMPORT_ERROR: Optional[Exception] = None\n_LIBROSA_MODULE: Any = None\n_LIBROSA_IMPORT_ERROR: Optional[Exception] = None\n_PEDALBOARD_MODULE: Any = None\n_PEDALBOARD_IMPORT_ERROR: Optional[Exception] = None\n_OMNIVOICE_CLASSES: Optional[Tuple[Any, Any]] = None\n_OMNIVOICE_IMPORT_ERROR: Optional[Exception] = None\n\n\ndef _get_torch_module() -> Any:\n    global _TORCH_MODULE, _TORCH_IMPORT_ERROR\n    if _TORCH_MODULE is not None:\n        return _TORCH_MODULE\n    if _TORCH_IMPORT_ERROR is not None:\n        raise RuntimeError(f"torch is unavailable in this environment: {_TORCH_IMPORT_ERROR}") from _TORCH_IMPORT_ERROR\n    try:\n        import torch as torch_module\n    except Exception as exc:\n        _TORCH_IMPORT_ERROR = exc\n        raise RuntimeError(f"torch is unavailable in this environment: {exc}") from exc\n    _TORCH_MODULE = torch_module\n    return _TORCH_MODULE\n\n\ndef _get_librosa_module() -> Any:\n    global _LIBROSA_MODULE, _LIBROSA_IMPORT_ERROR\n    if _LIBROSA_MODULE is not None:\n        return _LIBROSA_MODULE\n    if _LIBROSA_IMPORT_ERROR is not None:\n        raise RuntimeError(f"librosa is unavailable in this environment: {_LIBROSA_IMPORT_ERROR}") from _LIBROSA_IMPORT_ERROR\n    try:\n        import librosa as librosa_module\n    except Exception as exc:\n        _LIBROSA_IMPORT_ERROR = exc\n        raise RuntimeError(f"librosa is unavailable in this environment: {exc}") from exc\n    _LIBROSA_MODULE = librosa_module\n    return _LIBROSA_MODULE\n\n\ndef _get_pedalboard_pitch_shift() -> Any:\n    global _PEDALBOARD_MODULE, _PEDALBOARD_IMPORT_ERROR\n    if _PEDALBOARD_MODULE is not None:\n        return _PEDALBOARD_MODULE\n    if _PEDALBOARD_IMPORT_ERROR is not None:\n        raise RuntimeError(f"pedalboard is unavailable in this environment: {_PEDALBOARD_IMPORT_ERROR}") from _PEDALBOARD_IMPORT_ERROR\n    try:\n        from pedalboard import Pedalboard, PitchShift\n    except Exception as exc:\n        _PEDALBOARD_IMPORT_ERROR = exc\n        raise RuntimeError(f"pedalboard is unavailable in this environment: {exc}") from exc\n    _PEDALBOARD_MODULE = (Pedalboard, PitchShift)\n    return _PEDALBOARD_MODULE\n\n\ndef _get_omnivoice_classes() -> Tuple[Any, Any]:\n    global _OMNIVOICE_CLASSES, _OMNIVOICE_IMPORT_ERROR\n    if _OMNIVOICE_CLASSES is not None:\n        return _OMNIVOICE_CLASSES\n    if _OMNIVOICE_IMPORT_ERROR is not None:\n        raise RuntimeError(f"omnivoice is unavailable in this environment: {_OMNIVOICE_IMPORT_ERROR}") from _OMNIVOICE_IMPORT_ERROR\n    try:\n        from omnivoice import OmniVoice as omnivoice_cls\n    except Exception:\n        try:\n            from omnivoice.models.omnivoice import OmniVoice as omnivoice_cls\n        except Exception as exc:\n            _OMNIVOICE_IMPORT_ERROR = exc\n            raise RuntimeError(f"omnivoice is unavailable in this environment: {exc}") from exc\n    try:\n        from omnivoice.models.omnivoice import VoiceClonePrompt as voice_clone_prompt_cls\n    except Exception:\n        voice_clone_prompt_cls = None\n    _OMNIVOICE_CLASSES = (omnivoice_cls, voice_clone_prompt_cls)\n    return _OMNIVOICE_CLASSES\n\n\ndef _apply_output_peak_guard(audio_np: np.ndarray, target_peak: float = 0.92) -> tuple[np.ndarray, float]:\n    if audio_np.size == 0:\n        return audio_np.astype(np.float32), 0.0\n    peak = float(np.max(np.abs(audio_np)))\n    if peak <= 0.0 or peak <= float(target_peak):\n        return audio_np.astype(np.float32), 0.0\n    attenuation = float(target_peak) / peak\n    attenuation_db = 20.0 * math.log10(max(attenuation, 1e-8))\n    return (audio_np * attenuation).astype(np.float32), attenuation_db\n\ndef _normalize_device_name(value: Optional[str]) -> Optional[str]:\n    if value is None:\n        return None\n    normalized = str(value).strip().lower()\n    if not normalized or normalized == "auto":\n        return None\n    if normalized not in {"cpu", "cuda", "mps"}:\n        raise ValueError("OMNIVOICE_DEVICE must be one of: auto, cpu, cuda, mps")\n    return normalized\n\ndef _is_device_available(device: str) -> bool:\n    if device == "cpu":\n        return True\n    torch = _get_torch_module()\n    if device == "cpu":\n        return True\n    if device == "cuda":\n        return bool(torch.cuda.is_available())\n    if device == "mps":\n        return bool(hasattr(torch.backends, "mps") and torch.backends.mps.is_available())\n    return False\n\ndef get_best_device() -> str:\n    forced = _normalize_device_name(os.getenv("OMNIVOICE_DEVICE"))\n    if forced:\n        if not _is_device_available(forced):\n            raise RuntimeError(f"OMNIVOICE_DEVICE={forced} was requested but is not available in this environment")\n        return forced\n    try:\n        torch = _get_torch_module()\n    except RuntimeError:\n        return "cpu"\n    if torch.cuda.is_available():\n        return "cuda"\n    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():\n        return "mps"\n    return "cpu"\n\ndef _candidate_hf_cache_roots() -> list[Path]:\n    candidates: list[Path] = []\n    for env_key in ("MODEL_LOCAL_PATH", "HF_HOME", "TRANSFORMERS_CACHE"):\n        value = os.getenv(env_key)\n        if value:\n            candidates.append(Path(value))\n    candidates.extend(\n        [\n            Path("/runpod-volume/huggingface-cache"),\n            Path("/workspace/huggingface-cache"),\n            Path.home() / ".cache" / "huggingface",\n        ]\n    )\n\n    seen: set[str] = set()\n    unique: list[Path] = []\n    for path in candidates:\n        key = str(path)\n        if key in seen:\n            continue\n        seen.add(key)\n        unique.append(path)\n    return unique\n\ndef resolve_hf_cached_model_path(model_id: str) -> Optional[str]:\n    if "/" not in model_id:\n        return None\n    org, name = model_id.split("/", 1)\n    for cache_root in _candidate_hf_cache_roots():\n        root = cache_root / "hub" / f"models--{org}--{name}"\n        refs_main = root / "refs" / "main"\n        if refs_main.exists():\n            try:\n                snapshot_hash = refs_main.read_text(encoding="utf-8").strip()\n                snapshot_path = root / "snapshots" / snapshot_hash\n                if snapshot_path.exists():\n                    return str(snapshot_path)\n            except Exception:\n                pass\n        snapshots = sorted(glob.glob(str(root / "snapshots" / "*")))\n        if snapshots:\n            return snapshots[-1]\n    return None\n\ndef resolve_model_source(model_id: str) -> str:\n    explicit = os.getenv("MODEL_LOCAL_PATH")\n    if explicit and Path(explicit).exists():\n        return explicit\n    cached = resolve_hf_cached_model_path(model_id)\n    if cached:\n        return cached\n    return model_id\n\ndef infer_mode(reference_audio_path: Optional[str], custom_instruct: Optional[str], explicit_mode: Optional[str]) -> str:\n    if explicit_mode in {"clone", "design", "auto"}:\n        return explicit_mode\n    if reference_audio_path:\n        return "clone"\n    if custom_instruct and custom_instruct.strip():\n        return "design"\n    return "auto"\n\nclass OmniVoiceService:\n    def __init__(\n        self,\n        model_id: str = "k2-fsa/OmniVoice",\n        output_dir: Optional[str] = None,\n        prompt_cache_dir: Optional[str] = None,\n    ) -> None:\n        self.model_id = model_id\n        self.model_source = resolve_model_source(model_id)\n        self.device = get_best_device()\n        self.dtype: Any = None\n        self.output_dir = Path(output_dir or os.getenv("OUTPUT_DIR", "/runpod-volume/outputs" if Path("/runpod-volume").exists() else "/tmp/outputs"))\n        self.prompt_cache_dir = Path(prompt_cache_dir or os.getenv("PROMPT_CACHE_DIR", "/runpod-volume/prompt-cache" if Path("/runpod-volume").exists() else "/tmp/prompt-cache"))\n        ensure_dir(self.output_dir)\n        ensure_dir(self.prompt_cache_dir)\n        self.model: Optional[Any] = None\n        self._prompt_cache: Dict[str, Any] = {}\n        self._cache_file = self.prompt_cache_dir / "voice_clone_prompt_cache.json"\n        self._load_cache_from_disk()\n\n    def load_model(self) -> Any:\n        if self.model is None:\n            torch = _get_torch_module()\n            omnivoice_cls, _ = _get_omnivoice_classes()\n            self.dtype = torch.float16 if self.device != "cpu" else torch.float32\n            self.model = omnivoice_cls.from_pretrained(\n                self.model_source,\n                dtype=self.dtype,\n            )\n            if hasattr(self.model, "to"):\n                self.model = self.model.to(self.device)\n        return self.model\n\n    @property\n    def sampling_rate(self) -> int:\n        model = self.load_model()\n        return int(getattr(model, "sampling_rate", DEFAULT_TARGET_SR))\n\n    def _load_cache_from_disk(self) -> None:\n        if self._cache_file.exists():\n            try:\n                self._prompt_cache = json.loads(self._cache_file.read_text(encoding="utf-8"))\n            except Exception:\n                self._prompt_cache = {}\n\n    def _save_cache_to_disk(self) -> None:\n        tmp_file = self._cache_file.with_suffix(".tmp")\n        tmp_file.write_text(json.dumps(self._prompt_cache, ensure_ascii=False, indent=2), encoding="utf-8")\n        tmp_file.replace(self._cache_file)\n\n    @staticmethod\n    def _file_fingerprint(file_path: str) -> str:\n        p = Path(file_path)\n        stat = p.stat()\n        hasher = hashlib.md5()\n        with p.open("rb") as f:\n            while True:\n                chunk = f.read(1024 * 1024)\n                if not chunk:\n                    break\n                hasher.update(chunk)\n        # We exclude mtime to allow cache hits for re-processed files with same content\n        return f"{stat.st_size}:{hasher.hexdigest()}"\n\n    def _cache_key(self, ref_audio: str, ref_text: Optional[str]) -> str:\n        raw = f"{self._file_fingerprint(ref_audio)}::{ref_text or \'\'}"\n        return hashlib.md5(raw.encode("utf-8")).hexdigest()\n\n    def _prompt_to_serializable(self, prompt: Any) -> Dict[str, Any]:\n        ref_audio_tokens = getattr(prompt, "ref_audio_tokens", None)\n        ref_text = getattr(prompt, "ref_text", None)\n        ref_rms = getattr(prompt, "ref_rms", None)\n\n        if ref_audio_tokens is None:\n            raise ValueError("voice clone prompt has no ref_audio_tokens")\n\n        if hasattr(ref_audio_tokens, "cpu"):\n            ref_audio_tokens = ref_audio_tokens.cpu().tolist()\n\n        if isinstance(ref_rms, (np.floating, np.integer)):\n            ref_rms = float(ref_rms)\n        elif hasattr(ref_rms, "item"):\n            ref_rms = float(ref_rms.item())\n\n        return {\n            "ref_audio_tokens": ref_audio_tokens,\n            "ref_text": ref_text,\n            "ref_rms": ref_rms,\n        }\n\n    def _get_cached_prompt(self, key: str) -> Optional[Any]:\n        item = self._prompt_cache.get(key)\n        if item is None:\n            return None\n        torch = _get_torch_module()\n        _, voice_clone_prompt_cls = _get_omnivoice_classes()\n        if voice_clone_prompt_cls is None:\n            return None\n        try:\n            return voice_clone_prompt_cls(\n                ref_audio_tokens=torch.tensor(item["ref_audio_tokens"], dtype=torch.long),\n                ref_text=item.get("ref_text"),\n                ref_rms=item.get("ref_rms"),\n            )\n        except Exception:\n            return None\n\n    def create_voice_clone_prompt(\n        self,\n        ref_audio: str,\n        ref_text: Optional[str],\n        preprocess_prompt: bool = True,\n        language: Optional[str] = None,\n    ) -> Any:\n        model = self.load_model()\n        key = self._cache_key(ref_audio, ref_text)\n        cached = self._get_cached_prompt(key)\n        if cached is not None:\n            return cached\n\n        create_prompt_sig = inspect.signature(model.create_voice_clone_prompt)\n        prompt_kwargs = {\n            "ref_audio": ref_audio,\n            "ref_text": ref_text,\n            "preprocess_prompt": preprocess_prompt,\n        }\n        if "language" in create_prompt_sig.parameters and language is not None:\n            prompt_kwargs["language"] = language\n\n        prompt = model.create_voice_clone_prompt(**prompt_kwargs)\n        try:\n            self._prompt_cache[key] = self._prompt_to_serializable(prompt)\n            self._save_cache_to_disk()\n        except Exception:\n            pass\n        return prompt\n\n    def _apply_pitch_shift(self, audio_np: np.ndarray, pitch_shift: float) -> np.ndarray:\n        if pitch_shift is None or abs(float(pitch_shift) - 1.0) < 1e-6:\n            return audio_np.astype(np.float32)\n        ratio = max(0.5, min(2.0, float(pitch_shift)))\n        semitones = 12.0 * math.log2(ratio)\n        try:\n            pedalboard_cls, pitch_shift_cls = _get_pedalboard_pitch_shift()\n            board = pedalboard_cls([pitch_shift_cls(semitones=semitones)])\n            shifted = board(audio_np.reshape(1, -1).astype(np.float32), self.sampling_rate).flatten()\n            return shifted.astype(np.float32)\n        except RuntimeError:\n            librosa = _get_librosa_module()\n            shifted = librosa.effects.pitch_shift(audio_np.astype(np.float32), sr=self.sampling_rate, n_steps=semitones)\n            return shifted.astype(np.float32)\n\n    def get_prompt_voice_healthcheck(self) -> Dict[str, Any]:\n        return build_prompt_voice_healthcheck()\n\n    def _resolve_preset_reference(\n        self,\n        *,\n        voice_preset: Optional[str],\n        language: str,\n        reference_audio_path: Optional[str],\n        reference_audio_url: Optional[str],\n        reference_audio_base64: Optional[str],\n        ref_text: Optional[str],\n        emotion: str,\n        ad_emphasis: str,\n        speed: Optional[float],\n        pitch_shift: Optional[float],\n        num_step: Optional[int],\n        guidance_scale: Optional[float],\n    ) -> Dict[str, Any]:\n        preset = get_voice_preset(voice_preset) if voice_preset else None\n        if voice_preset and not preset:\n            raise ValueError(f"Unknown voice_preset: {voice_preset}")\n\n        if preset and language and canonical_lang(language) != preset["language"]:\n            raise ValueError(\n                f"voice_preset \'{preset[\'preset_key\']}\' belongs to language={preset[\'language\']}, not {canonical_lang(language)}"\n            )\n\n        resolved_reference_audio_path = reference_audio_path\n        resolved_ref_text = ref_text\n        resolved_emotion = emotion\n        resolved_ad_emphasis = ad_emphasis\n        preset_meta: Optional[Dict[str, Any]] = None\n        has_manual_reference = any(\n            bool(item)\n            for item in (reference_audio_path, reference_audio_url, reference_audio_base64)\n        )\n        manual_overrides = {\n            "speed": speed,\n            "pitch_shift": pitch_shift,\n            "num_step": num_step,\n            "guidance_scale": guidance_scale,\n        }\n\n        if preset:\n            preset_path = Path(preset["audio_path"])\n            using_preset_audio = not has_manual_reference\n            if using_preset_audio and resolved_reference_audio_path is None:\n                if not preset_path.exists():\n                    raise FileNotFoundError(\n                        f"voice_preset \'{preset[\'preset_key\']}\' expects audio at {preset_path}, but file was not found"\n                    )\n                resolved_reference_audio_path = str(preset_path)\n\n            if using_preset_audio and not resolved_ref_text and preset.get("reference_text"):\n                resolved_ref_text = str(preset["reference_text"])\n\n            preset_meta = {\n                "preset_key": preset["preset_key"],\n                "label": preset["label"],\n                "audio_path": preset["audio_path"],\n                "reference_text_path": preset["reference_text_path"],\n                "reference_text_exists": preset["reference_text_exists"],\n                "style_tags": preset.get("style_tags", []),\n                "source": "preset_audio" if using_preset_audio else "manual_reference_override",\n                "selection": {\n                    "emotion": resolved_emotion,\n                    "ad_emphasis": resolved_ad_emphasis,\n                    "manual_overrides": manual_overrides,\n                },\n            }\n\n        return {\n            "reference_audio_path": resolved_reference_audio_path,\n            "ref_text": resolved_ref_text,\n            "emotion": resolved_emotion,\n            "ad_emphasis": resolved_ad_emphasis,\n            "manual_overrides": manual_overrides,\n            "preset_meta": preset_meta,\n        }\n\n    def synthesize(\n        self,\n        *,\n        text: str,\n        language: str,\n        mode: Optional[str] = None,\n        reference_audio_path: Optional[str] = None,\n        reference_audio_url: Optional[str] = None,\n        reference_audio_base64: Optional[str] = None,\n        voice_preset: Optional[str] = None,\n        ref_text: Optional[str] = None,\n        emotion: str = "Mặc định",\n        ad_emphasis: str = "Không bổ trợ",\n        ad_safe: bool = True,\n        custom_instruct: Optional[str] = None,\n        speed: Optional[float] = None,\n        pitch_shift: Optional[float] = None,\n        num_step: Optional[int] = None,\n        guidance_scale: Optional[float] = None,\n        output_filename: Optional[str] = None,\n        return_base64: bool = False,\n        save_output: bool = True,\n        debug: bool = False,\n        preprocess_reference: bool = False,\n        ref_trim_silence: bool = True,\n        ref_trim_top_db: int = 35,\n        ref_apply_vad: bool = True,\n        ref_vad_top_db: int = 32,\n        ref_max_internal_silence_ms: int = 120,\n        ref_apply_denoise: bool = False,\n        ref_denoise_strength: float = 0.18,\n        ref_apply_rms_normalize: bool = True,\n        ref_target_rms_dbfs: float = -22.0,\n        ref_min_seconds: float = 1.5,\n        ref_max_seconds: float = 10.0,\n        work_dir: Optional[str] = None,\n    ) -> Dict[str, Any]:\n        language = canonical_lang(language)\n        preset_resolution = self._resolve_preset_reference(\n            voice_preset=voice_preset,\n            language=language,\n            reference_audio_path=reference_audio_path,\n            reference_audio_url=reference_audio_url,\n            reference_audio_base64=reference_audio_base64,\n            ref_text=ref_text,\n            emotion=emotion,\n            ad_emphasis=ad_emphasis,\n            speed=speed,\n            pitch_shift=pitch_shift,\n            num_step=num_step,\n            guidance_scale=guidance_scale,\n        )\n        reference_audio_path = preset_resolution["reference_audio_path"]\n        ref_text = preset_resolution["ref_text"]\n        emotion = preset_resolution["emotion"]\n        ad_emphasis = preset_resolution["ad_emphasis"]\n        manual_overrides = preset_resolution["manual_overrides"]\n        preset_meta = preset_resolution["preset_meta"]\n\n        cfg = get_effective_config(language, emotion, ad_emphasis, manual_overrides)\n        cfg = clamp_prosody(language, cfg)\n        effective_gain_db = safe_gain_for_style(language, emotion, ad_emphasis, enabled=ad_safe)\n        work_dir = work_dir or tempfile.mkdtemp(prefix="omnivoice-job-")\n        ensure_dir(work_dir)\n        model = self.load_model()\n        started_at = time.perf_counter()\n\n        normalized_text = preprocess_text_for_tts(text, language, is_reference=False)\n        if not normalized_text:\n            raise ValueError("text is empty after normalization")\n\n        source_ref = resolve_reference_audio_source(\n            reference_audio_path=reference_audio_path,\n            reference_audio_url=reference_audio_url,\n            reference_audio_base64=reference_audio_base64,\n            work_dir=work_dir,\n        )\n\n        reference_meta: Dict[str, Any] = {\n            "provided": bool(source_ref),\n            "source_path": str(source_ref) if source_ref else None,\n            "preprocessed_path": None,\n            "voice_preset": preset_meta,\n        }\n\n        cleaned_ref_text = build_ref_text(ref_text, normalized_text, language)\n        reference_quality = reference_quality_note(language, cleaned_ref_text)\n        if source_ref and preprocess_reference:\n            preprocessed_ref = Path(work_dir) / "reference_preprocessed.wav"\n            reference_stats = preprocess_reference_audio(\n                source_path=source_ref,\n                output_path=preprocessed_ref,\n                target_sr=self.sampling_rate,\n                trim_silence=ref_trim_silence,\n                trim_top_db=ref_trim_top_db,\n                apply_vad=ref_apply_vad,\n                vad_top_db=ref_vad_top_db,\n                max_internal_silence_ms=ref_max_internal_silence_ms,\n                apply_denoise=manual_overrides.get("ref_apply_denoise", True),\n                denoise_strength=ref_denoise_strength,\n                apply_rms_normalize=ref_apply_rms_normalize,\n                target_rms_dbfs=ref_target_rms_dbfs,\n                min_seconds=ref_min_seconds,\n                max_seconds=ref_max_seconds,\n            )\n            reference_meta.update(reference_stats)\n            source_ref = preprocessed_ref\n        elif source_ref:\n            reference_meta["preprocessed_path"] = str(source_ref)\n\n        actual_mode = infer_mode(str(source_ref) if source_ref else None, custom_instruct, mode)\n        voice_clone_prompt = None\n        if actual_mode == "clone":\n            if source_ref is None:\n                raise ValueError("mode=\'clone\' requires reference audio")\n            voice_clone_prompt = self.create_voice_clone_prompt(\n                ref_audio=str(source_ref),\n                ref_text=cleaned_ref_text,\n                preprocess_prompt=bool(cfg["preprocess_prompt"]),\n                language=language,\n            )\n\n        chunk_infos = segment_text_with_pauses(\n            normalized_text,\n            int(cfg["join_silence_ms"]),\n            int(cfg["max_segment_chars"]) if cfg.get("max_segment_chars") else None,\n            20 if canonical_lang(language) == "my" else None,\n            language,\n        )\n        if not chunk_infos:\n            chunk_infos = [{"text": normalized_text, "pause_ms": int(cfg["join_silence_ms"])}]\n\n        rendered_parts = []\n        for chunk_info in chunk_infos:\n            prepared_text = add_config_text_omni(chunk_info["text"])\n            gen_kwargs: Dict[str, Any] = {\n                "text": prepared_text,\n                "language": language,\n                "speed": float(cfg["speed"]),\n                "num_step": int(cfg["num_step"]),\n                "guidance_scale": float(cfg["guidance_scale"]),\n                "t_shift": float(cfg["t_shift"]),\n                "layer_penalty_factor": float(cfg["layer_penalty_factor"]),\n                "position_temperature": float(cfg["position_temperature"]),\n                "class_temperature": float(cfg["class_temperature"]),\n                "denoise": bool(cfg["denoise"]),\n                "preprocess_prompt": bool(cfg["preprocess_prompt"]),\n                "postprocess_output": bool(cfg["postprocess_output"]),\n                "audio_chunk_duration": float(cfg["audio_chunk_duration"]),\n                "audio_chunk_threshold": float(cfg["audio_chunk_threshold"]),\n            }\n\n            instruct_value = (custom_instruct or cfg.get("instruct") or "").strip()\n            if instruct_value:\n                gen_kwargs["instruct"] = instruct_value\n\n            if actual_mode == "clone" and voice_clone_prompt is not None:\n                gen_kwargs["voice_clone_prompt"] = voice_clone_prompt\n\n            torch = _get_torch_module()\n            with torch.inference_mode():\n                out = model.generate(**gen_kwargs)\n\n            piece = np.asarray(out[0], dtype=np.float32).flatten()\n            rendered_parts.append(\n                {\n                    "audio": piece,\n                    "pause_ms": max(int(cfg.get("min_join_silence_ms", 45)), int(chunk_info["pause_ms"])),\n                }\n            )\n\n        assembled_parts = []\n        for index, item in enumerate(rendered_parts):\n            if index > 0:\n                pause_ms = int(item["pause_ms"])\n                assembled_parts.append(np.zeros(int(self.sampling_rate * pause_ms / 1000.0), dtype=np.float32))\n            assembled_parts.append(item["audio"].astype(np.float32, copy=False))\n\n        combined = np.concatenate(assembled_parts, axis=0).astype(np.float32, copy=False)\n\n        combined = self._apply_pitch_shift(combined, float(cfg["pitch_shift"]))\n        trailing = np.zeros(int(self.sampling_rate * float(cfg["trailing_silence_ms"]) / 1000.0), dtype=np.float32)\n        combined = np.concatenate([combined, trailing]).astype(np.float32)\n\n        gain_ratio = 10 ** (effective_gain_db / 20.0)\n        final_audio = np.clip(combined * gain_ratio, -1.0, 1.0).astype(np.float32)\n\n        output_stub = hashlib.md5(\n            json.dumps(\n                {\n                    "text": normalized_text,\n                    "language": language,\n                    "mode": actual_mode,\n                    "voice_preset": voice_preset,\n                    "ref_text": cleaned_ref_text,\n                    "cfg": {\n                        "speed": float(cfg["speed"]),\n                        "pitch_shift": float(cfg["pitch_shift"]),\n                        "num_step": int(cfg["num_step"]),\n                        "guidance_scale": float(cfg["guidance_scale"]),\n                    },\n                },\n                ensure_ascii=False,\n                sort_keys=True,\n            ).encode("utf-8")\n        ).hexdigest()[:10]\n        unique_suffix = uuid.uuid4().hex[:8]\n        output_filename = output_filename or f"{language}_{actual_mode}_{output_stub}_{unique_suffix}.wav"\n        output_path = self.output_dir / output_filename\n        if save_output:\n            write_output_wav(final_audio, self.sampling_rate, output_path)\n\n        compute_seconds = round(time.perf_counter() - started_at, 4)\n        response = {\n            "ok": True,\n            "mode": actual_mode,\n            "language": language,\n            "language_label": LANGUAGE_LABELS.get(language, language),\n            "text": normalized_text,\n            "ref_text": cleaned_ref_text,\n            "reference_quality": reference_quality,\n            "segments": [item["text"] for item in chunk_infos],\n            "config": cfg,\n            "effective_config": {\n                "speed": float(cfg["speed"]),\n                "pitch_shift": float(cfg["pitch_shift"]),\n                "num_step": int(cfg["num_step"]),\n                "guidance_scale": float(cfg["guidance_scale"]),\n                "join_silence_ms": int(cfg["join_silence_ms"]),\n                "trailing_silence_ms": int(cfg["trailing_silence_ms"]),\n            },\n            "ad_safe": bool(ad_safe),\n            "reference": reference_meta,\n            "audio_base64": encode_wav_base64(final_audio, self.sampling_rate) if return_base64 else None,\n            "output_path": str(output_path) if save_output else None,\n            "sampling_rate": self.sampling_rate,\n            "duration_sec": round(len(final_audio) / self.sampling_rate, 4),\n            "compute_seconds": compute_seconds,\n            "gain_db": effective_gain_db,\n            "effective_gain_db": effective_gain_db,\n            "device": self.device,\n        }\n        if debug:\n            response["voice_library"] = self.get_prompt_voice_healthcheck()\n            response["model_source"] = self.model_source\n\n        try:\n            torch = _get_torch_module()\n        except RuntimeError:\n            torch = None\n        if torch is not None and torch.cuda.is_available():\n            torch.cuda.empty_cache()\n        gc.collect()\n\n        return response\n\ndef get_voice_library_status() -> Dict[str, Any]:\n    return get_service().get_prompt_voice_healthcheck()\n\nSERVICE: Optional[OmniVoiceService] = None\n\ndef get_service() -> OmniVoiceService:\n    global SERVICE\n    if SERVICE is None:\n        SERVICE = OmniVoiceService(model_id=os.getenv("MODEL_ID", "k2-fsa/OmniVoice"))\n    return SERVICE\n',
}

for filename, source in module_sources.items():
    (PKG_DIR / filename).write_text(source, encoding="utf-8")

if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

print("Created package at:", PKG_DIR)
print("Files:", sorted(p.name for p in PKG_DIR.glob("*.py")))


Created package at: /content/myanmar_voice_pipeline_src/myanmar_voice_pipeline
Files: ['__init__.py', 'audio_processing.py', 'engine.py', 'presets.py', 'text_normalization.py']


In [18]:
#@title 4) Import pipeline và cấu hình model
import os, sys, json, time, tempfile, shutil, gc
from pathlib import Path

os.environ["OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["PROMPT_CACHE_DIR"] = str(CACHE_DIR)  # cache embedding reference voice clone
os.environ["MODEL_ID"] = "k2-fsa/OmniVoice"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from myanmar_voice_pipeline.engine import OmniVoiceService
from myanmar_voice_pipeline.text_normalization import preprocess_text_for_tts, segment_text_with_pauses
from myanmar_voice_pipeline.presets import get_effective_config, clamp_prosody
from myanmar_voice_pipeline.audio_processing import preprocess_reference_audio, analyze_reference_audio, load_audio_mono

LANGUAGE = "my"

print("Pipeline imported.")
print("Model:", os.environ["MODEL_ID"])
print("Reference voice folder:", REF_DIR)


Pipeline imported.
Model: k2-fsa/OmniVoice
Reference voice folder: /content/drive/MyDrive/OmniVoice_Myanmar_Gradio/reference_voices


## 5) Thêm voice clone vào thư mục Drive

Bạn có 2 cách:

- Cách 1: Upload trực tiếp trong Gradio UI.
- Cách 2: Copy file `.wav/.mp3/.flac/.m4a/.ogg` vào thư mục:

`/content/drive/MyDrive/OmniVoice_Myanmar_Gradio/reference_voices`

Tên file nên dễ nhớ, ví dụ:

- `girl_clean_01.wav`
- `male_ads_01.wav`
- `myanmar_voice_test.wav`

Reference voice nên dài **3–10 giây**, 1 người nói, ít noise, không nhạc nền.


In [19]:
#@title 6) Chạy Gradio UI cho Myanmar Voice Clone
import gradio as gr
import numpy as np
import soundfile as sf
from IPython.display import display
from pathlib import Path
import os, json, shutil, tempfile, time, traceback, gc

AUDIO_EXTS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac", ".webm"}

_service = None

def get_service():
    global _service
    if _service is None:
        _service = OmniVoiceService(
            model_id=os.getenv("MODEL_ID", "k2-fsa/OmniVoice"),
            output_dir=str(OUTPUT_DIR),
            prompt_cache_dir=str(CACHE_DIR),
        )
        # Load ngay để lần generate đầu không bị cảm giác treo quá lâu
        _service.load_model()
    return _service

def list_voice_files():
    REF_DIR.mkdir(parents=True, exist_ok=True)
    files = []
    for p in sorted(REF_DIR.iterdir()):
        if p.is_file() and p.suffix.lower() in AUDIO_EXTS:
            files.append(p.name)
    return files

def refresh_voice_dropdown():
    choices = list_voice_files()
    value = choices[0] if choices else None
    return gr.update(choices=choices, value=value), f"Đã tìm thấy {len(choices)} voice trong Drive."

def safe_voice_filename(name: str, original_path: str):
    name = (name or "").strip()
    ext = Path(original_path).suffix.lower() or ".wav"
    if not name:
        name = Path(original_path).stem
    keep = []
    for ch in name:
        if ch.isalnum() or ch in ("-", "_", " "):
            keep.append(ch)
    clean = "".join(keep).strip().replace(" ", "_")
    if not clean:
        clean = f"voice_{int(time.time())}"
    if not clean.lower().endswith(ext):
        clean += ext
    return clean

def save_uploaded_voice(uploaded_audio, voice_name):
    if not uploaded_audio:
        choices = list_voice_files()
        return gr.update(choices=choices, value=(choices[0] if choices else None)), "Bạn chưa upload audio."
    src = Path(uploaded_audio)
    if not src.exists():
        choices = list_voice_files()
        return gr.update(choices=choices, value=(choices[0] if choices else None)), f"Không tìm thấy file upload: {src}"

    filename = safe_voice_filename(voice_name, str(src))
    dst = REF_DIR / filename
    shutil.copy2(src, dst)

    choices = list_voice_files()
    return gr.update(choices=choices, value=filename), f"Đã lưu voice vào Drive: {dst}"

def get_reference_path(selected_voice, uploaded_audio, prefer_uploaded):
    if prefer_uploaded and uploaded_audio:
        return str(uploaded_audio), "uploaded_audio"
    if selected_voice:
        p = REF_DIR / selected_voice
        if p.exists():
            return str(p), "drive_voice"
    if uploaded_audio:
        return str(uploaded_audio), "uploaded_audio"
    raise ValueError("Bạn cần chọn voice trong Drive hoặc upload reference audio.")

def preview_reference(selected_voice, uploaded_audio, prefer_uploaded):
    try:
        ref_path, source = get_reference_path(selected_voice, uploaded_audio, prefer_uploaded)
        audio, sr = load_audio_mono(ref_path, sr=None)
        meta = analyze_reference_audio(audio, sr)
        return ref_path, json.dumps({"source": source, "path": ref_path, "quality": meta}, ensure_ascii=False, indent=2)
    except Exception as e:
        return None, f"Lỗi preview reference: {e}"

def normalize_preview(text, join_silence_ms, max_segment_chars):
    try:
        normalized = preprocess_text_for_tts(text or "", "my", is_reference=False)
        chunks = segment_text_with_pauses(
            normalized,
            join_silence_ms=int(join_silence_ms),
            max_chars=int(max_segment_chars) if max_segment_chars else None,
            min_chars=8,
            lang="my",
        )
        return normalized, json.dumps(chunks, ensure_ascii=False, indent=2)
    except Exception as e:
        return "", f"Lỗi normalize/segment: {e}"

def generate_voice_clone(
    text,
    selected_voice,
    uploaded_audio,
    prefer_uploaded,
    reference_text,
    emotion,
    ad_emphasis,
    num_step,
    guidance_scale,
    speed,
    pitch_shift,
    join_silence_ms,
    trailing_silence_ms,
    max_segment_chars,
    preprocess_reference,
    ref_apply_vad,
    ref_vad_top_db,
    ref_apply_denoise,
    ref_denoise_strength,
    ref_apply_rms_normalize,
    ref_target_rms_dbfs,
):
    try:
        text = (text or "").strip()
        if not text:
            raise ValueError("Bạn chưa nhập text Myanmar cần đọc.")

        ref_path, ref_source = get_reference_path(selected_voice, uploaded_audio, prefer_uploaded)
        ref_text = (reference_text or "").strip() or None

        service = get_service()

        # Manual overrides để giữ clone-only nhưng vẫn cho chỉnh prosody/render
        output_name = f"myanmar_clone_{int(time.time())}.wav"

        result = service.synthesize(
            text=text,
            language="my",
            mode="clone",
            reference_audio_path=ref_path,
            ref_text=ref_text,
            emotion=emotion,
            ad_emphasis=ad_emphasis,
            ad_safe=True,
            speed=float(speed),
            pitch_shift=float(pitch_shift),
            num_step=int(num_step),
            guidance_scale=float(guidance_scale),
            output_filename=output_name,
            return_base64=False,
            save_output=True,
            debug=True,
            preprocess_reference=bool(preprocess_reference),
            ref_trim_silence=True,
            ref_trim_top_db=35,
            ref_apply_vad=bool(ref_apply_vad),
            ref_vad_top_db=int(ref_vad_top_db),
            ref_max_internal_silence_ms=120,
            ref_apply_denoise=bool(ref_apply_denoise),
            ref_denoise_strength=float(ref_denoise_strength),
            ref_apply_rms_normalize=bool(ref_apply_rms_normalize),
            ref_target_rms_dbfs=float(ref_target_rms_dbfs),
            ref_min_seconds=1.5,
            ref_max_seconds=10.0,
            work_dir=str(WORK_DIR / f"job_{int(time.time())}"),
        )

        out_path = result.get("output_path")
        normalized = result.get("normalized_text", "")
        chunks = result.get("chunks", [])

        meta = {
            "reference_source": ref_source,
            "reference_path": ref_path,
            "output_path": out_path,
            "config": result.get("config"),
            "reference": result.get("reference"),
            "timing": result.get("timing"),
            "chunks": chunks,
            "warnings": result.get("warnings"),
        }

        return out_path, normalized, json.dumps(meta, ensure_ascii=False, indent=2)

    except Exception as e:
        err = traceback.format_exc()
        return None, "", f"ERROR: {e}\n\n{err}"

def unload_model():
    global _service
    _service = None
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    return "Đã unload model và dọn GPU cache."

initial_choices = list_voice_files()
initial_value = initial_choices[0] if initial_choices else None

with gr.Blocks(title="Myanmar Voice Clone Only", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Myanmar Voice Clone Only

    Chọn/upload **reference voice**, nhập text Myanmar, chỉnh thông số rồi bấm **Generate**.

    Không dùng prompt voice preset. Voice ở đây là **file giọng mẫu thật** trong Google Drive hoặc file bạn upload.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("## 1) Chọn voice reference")
            selected_voice = gr.Dropdown(
                choices=initial_choices,
                value=initial_value,
                label="Voice trong Google Drive",
                info="Các file nằm trong thư mục reference_voices.",
            )
            refresh_btn = gr.Button("Reload danh sách voice")
            refresh_status = gr.Textbox(label="Trạng thái", interactive=False)

            uploaded_audio = gr.Audio(
                label="Hoặc upload reference voice mới",
                type="filepath",
                sources=["upload", "microphone"],
            )
            prefer_uploaded = gr.Checkbox(
                value=True,
                label="Ưu tiên dùng audio vừa upload khi generate",
            )
            voice_name = gr.Textbox(
                label="Tên lưu voice vào Drive",
                placeholder="VD: myanmar_female_clean_01",
            )
            save_voice_btn = gr.Button("Lưu uploaded voice vào Drive")

            preview_btn = gr.Button("Kiểm tra reference voice")
            ref_audio_preview = gr.Audio(label="Reference đang dùng", type="filepath")
            ref_meta = gr.Textbox(label="Thông tin reference", lines=10)

        with gr.Column(scale=2):
            gr.Markdown("## 2) Text và thông số clone")
            text_input = gr.Textbox(
                label="Text Myanmar cần đọc",
                lines=6,
                value="ဒီနေ့ကစပြီး လည်ပင်းပခုံးနာကျင်မှု လက် ခြေထုံခြင်း အဆစ်အမြစ်နာကျင်မှုတွေကို စိတ်ပူစရာမလိုတော့ဘဲ B1ကို နေ့စဉ်သောက်သုံးရုံနဲ့ အရိုးနဲ့အဆစ်ကျန်းမာရေးကို ထောက်ပံ့ပေးနိုင်ပြီတစ်ပတ် အကြာမှာ ထုံကျင်မှုတွေ  သက်သာလာ‌ပြီး တစ်လခန့်အကြာမှာတော့ ကြွက်သားတောင့်တင်းမှု အဆစ်နာကျင်မှုနဲ့ ရောင်ရမ်းမှုတွေကို  လျော့နည်းလာ‌စေမှာဖြစ်ပါတယ်အသက်ကြီးပြီးနောက်ပိုင်းမှာလည်း အဆစ်များ ပျော့ပျောင်းပြီး လှုပ်ရှားမှုကောင်းမွန်စေဖို့ကိုလည်း ကူညီပေးပါတယ်",
            )
            reference_text = gr.Textbox(
                label="Ref text / transcript của reference voice",
                lines=4,
                value="ဆိုတော့ လိုအပ်ချက်ကတော့ အရိုးအဆစ် ကျန်းမာရေးအတွက် အထူးထုတ်လုပ်ထားတဲ့ Calcium Gold Colostrum Milk ကို နေ့စဉ် သောက်သုံးပေးရုံပါပဲ။",
                placeholder="Paste transcript đúng của voice mẫu vào đây. Có thể để trống nếu không có.",
                interactive=True,
            )

            with gr.Accordion("Thông số chính", open=True):
                with gr.Row():
                    num_step = gr.Slider(24, 48, value=36, step=1, label="Num step")
                    guidance_scale = gr.Slider(1.0, 5.0, value=4.0, step=0.05, label="Guidance")
                with gr.Row():
                    speed = gr.Slider(0.85, 2.0, value=1.0, step=0.005, label="Speed")
                    pitch_shift = gr.Slider(0.92, 1.08, value=1.0, step=0.005, label="Pitch")
                with gr.Row():
                    join_silence_ms = gr.Slider(50, 220, value=95, step=5, label="Join silence ms")
                    trailing_silence_ms = gr.Slider(100, 500, value=180, step=5, label="Trailing silence ms")
                    max_segment_chars = gr.Slider(40, 160, value=90, step=5, label="Max segment chars")

            with gr.Accordion("Style nhẹ / cảm xúc", open=False):
                emotion = gr.Dropdown(
                    ["Mặc định", "Vui vẻ (Happy)", "Buồn bã (Sad)", "Hào hứng (Excited)", "Giận dữ (Angry)", "Nhẹ nhàng (Gentle)"],
                    value="Mặc định",
                    label="Emotion",
                )
                ad_emphasis = gr.Dropdown(
                    ["Không bổ trợ", "Cường điệu rất nhẹ", "Cường điệu nhẹ", "Cường điệu vừa", "Cường điệu mạnh"],
                    value="Không bổ trợ",
                    label="Ad emphasis",
                )

            with gr.Accordion("Preprocess reference audio", open=False):
                preprocess_reference = gr.Checkbox(value=True, label="Bật preprocess reference")
                ref_apply_vad = gr.Checkbox(value=True, label="VAD / cắt khoảng lặng")
                ref_vad_top_db = gr.Slider(24, 45, value=32, step=1, label="VAD top_db")
                ref_apply_denoise = gr.Checkbox(value=False, label="Denoise nhẹ")
                ref_denoise_strength = gr.Slider(0.05, 0.35, value=0.18, step=0.01, label="Denoise strength")
                ref_apply_rms_normalize = gr.Checkbox(value=True, label="RMS normalize")
                ref_target_rms_dbfs = gr.Slider(-30, -16, value=-22, step=0.5, label="Target RMS dBFS")

            with gr.Row():
                normalize_btn = gr.Button("Preview normalize/segment")
                generate_btn = gr.Button("Generate clone voice", variant="primary")
                unload_btn = gr.Button("Unload model")

            normalized_text = gr.Textbox(label="Normalized text", lines=3)
            segment_preview = gr.Textbox(label="Segments / Metadata", lines=14)
            output_audio = gr.Audio(label="Output clone voice", type="filepath")

    refresh_btn.click(
        refresh_voice_dropdown,
        inputs=[],
        outputs=[selected_voice, refresh_status],
    )

    save_voice_btn.click(
        save_uploaded_voice,
        inputs=[uploaded_audio, voice_name],
        outputs=[selected_voice, refresh_status],
    )

    preview_btn.click(
        preview_reference,
        inputs=[selected_voice, uploaded_audio, prefer_uploaded],
        outputs=[ref_audio_preview, ref_meta],
    )

    normalize_btn.click(
        normalize_preview,
        inputs=[text_input, join_silence_ms, max_segment_chars],
        outputs=[normalized_text, segment_preview],
    )

    generate_btn.click(
        generate_voice_clone,
        inputs=[
            text_input,
            selected_voice,
            uploaded_audio,
            prefer_uploaded,
            reference_text,
            emotion,
            ad_emphasis,
            num_step,
            guidance_scale,
            speed,
            pitch_shift,
            join_silence_ms,
            trailing_silence_ms,
            max_segment_chars,
            preprocess_reference,
            ref_apply_vad,
            ref_vad_top_db,
            ref_apply_denoise,
            ref_denoise_strength,
            ref_apply_rms_normalize,
            ref_target_rms_dbfs,
        ],
        outputs=[output_audio, normalized_text, segment_preview],
    )

    unload_btn.click(unload_model, inputs=[], outputs=[refresh_status])

demo.queue(max_size=10)
demo.launch(share=True, debug=True)


/tmp/ipykernel_25936/557584911.py:208: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Myanmar Voice Clone Only", theme=gr.themes.Soft()) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7f1367c6398a4d4ad0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7f1367c6398a4d4ad0.gradio.live


## Gợi ý chỉnh khi test

- Nếu có tiếng ngắt kiểu “è/ừ”: thử giảm `Guidance` từ `4.0` xuống `3.8–3.9`, giữ `Num step = 36`.
- Nếu phát âm chưa rõ chữ: thử tăng `Guidance` lên `4.1–4.25` hoặc `Num step = 40`.
- Nếu giọng bị robotic/cứng: giảm `Guidance`, giữ `Speed = 1.0; có thể kéo tối đa 2.0 nếu cần nhanh hơn`, tắt `Ad emphasis`.
- Nếu câu dài bị vỡ nhịp: giảm `Max segment chars` xuống `70–80`.
- Nếu nối câu quá gấp: tăng `Join silence ms` lên `110–130`.
- Reference voice nên sạch, 3–10 giây, một người nói, không nhạc nền.
